# Evaluation visualization

Read completed evaluation outputs, prepare categorical and Anchor-Q summaries, and save six figures and readable tables. No weights, inference, extraction or metric scoring are needed. Run cells in order in a fresh kernel. Numerical and presentation runs are saved separately. Figures are exported as PNG at 300 dpi and EPS.


## 1. Parameters

Choose the input outputs directory and a new presentation ID. Shared fonts, dimensions and figure names follow the embedded project protocol.


In [ ]:
from pathlib import Path


def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()

# The directory containing contextual-anchor-q/, contextual-dimsim/ and contextual-random-control/.
EVALUATION_OUTPUT_ROOT = PROJECT_ROOT / "outputs"

# None selects the only completed run per analysis. Multiple completed runs require explicit IDs.
# EVALUATION_RUN_IDS = {
#     "contextual-anchor-q": "evaluation-01",
#     "contextual-dimsim": "evaluation-01",
#     "contextual-random-control": "evaluation-01",
# }
EVALUATION_RUN_IDS = None

# Results are written to OUTPUT_PROJECT_ROOT/outputs/contextual-evaluation-report/PRESENTATION_ID/.
OUTPUT_PROJECT_ROOT = PROJECT_ROOT / "outputs"
PRESENTATION_ID = "duyu-chinese-lessons-zh-cn-v1"

# Leave empty when Arial is installed. Otherwise provide existing Arial regular and bold font files.
FONT_FILES = []
# FONT_FILES = [Path("/kaggle/input/your-fonts/arial.ttf"), Path("/kaggle/input/your-fonts/arialbd.ttf")]

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["numpy", "matplotlib"]

# Exact evaluation bundle containing occurrences, sentences and the fixed balanced sample.
BUNDLE_PATH = Path("EDIT_ME_EVALUATION_BUNDLE_PATH")

BUNDLE_MANIFEST_SHA256 = "EDIT_ME_BUNDLE_MANIFEST_SHA256"

# prepare: compute from saved scores; reuse: render exact existing derived runs.
DERIVED_MODE = "prepare"

DERIVED_RUN_IDS = {"interpretation": "evaluation-interpretation-01", "presentation_data": "evaluation-data-01"}


## 2. Dependencies

Only NumPy and Matplotlib are required beyond the Python standard library. Package installation is optional.


In [ ]:
from __future__ import annotations

import hashlib
import copy
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
from contextlib import contextmanager
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any, Iterable
from numbers import Real
from statistics import NormalDist

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer is required")
import tomllib

if "EDIT_ME" in str(EVALUATION_OUTPUT_ROOT):
    raise ValueError("Set EVALUATION_OUTPUT_ROOT in the parameter cell")
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

_plot_cache = tempfile.TemporaryDirectory(prefix="evaluation-matplotlib-")
os.environ["MPLCONFIGDIR"] = _plot_cache.name
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

print("numpy", np.__version__)
print("matplotlib", matplotlib.__version__)


In [ ]:
BUILD_METADATA = {
    "schema_version": 1,
    "scope": "evaluation_visualization",
    "source_sha256": {
        "objectives.py": "2cdb95c3044103e6138d28038d696e43b200ea7b0e94c3131cac9a3fd3c0df47",
        "evaluation/anchor_sampling.py": "b1eed3343e60be7b068e4468df31827d892c66cd52a7a7842197506467dc52e8",
        "evaluation/anchor_q_reporting.py": "a9ff7e0a7d57e015e8d5068d1d6f8440f0883d20e27b681c964144c28487fcf5",
        "evaluation/interpretation.py": "6cfed6d565bec8c407781aa04c4c02eb98d25543117876e45b33fafcfa2f24bf",
        "research/artifacts.py": "663ec77c515bcbe1f350aa12b2d4fea80491ef10512069be387845f8c522c239",
        "evaluation/evaluation_outputs.py": "7c7c5fce3bba3026d967abb91b0325295ed88494add2337c827ac74740ffaa5f",
        "evaluation/interpretation_outputs.py": "380c3081dc65746b04cf6237e516d53bb0aaf017d701d472cb6935f4792a8705",
        "evaluation/interpretation_inputs.py": "6bdc1948cc74da4d4d4c58e30a99260392e5998145adf43032f4124d2faea87b",
        "evaluation/interpretation_reporting.py": "e7eae6cd175efd730b1f748e9b7138b1338ef2dd04b55c2e29f4e3ae5d38f858",
        "evaluation/evaluation_tables.py": "5e1c5c3ff81e15aff0d958b6f2f58afe2f1c711c090837935ce9cece0c83b638",
        "visualization/core.py": "ab70b4b4ff92482498c863ed090470f10e8bb38f24482b794ab4a1f1d9a5c325",
        "visualization/interpretation.py": "2e4475b8535187827f50b8da075a9ea7997743defc7dd4a12a8bd3b29c039dfa",
        "visualization/evaluation.py": "7209146408b4f94678d3e1b3983a5d375d5aab59ccdbc1f7c01634037d3c880e",
        "visualization/interpretation_report.py": "1f4df0ce0ad321ba89bd07810b84f18c09fa7c973fc1547086324cb86b3022ed",
        "visualization/evaluation_report.py": "90b710badae365c22ee2a6f4a2621868d33d8dd9f87ff1987b7bcf5b555f5870",
        "visualization/evaluation_workflow.py": "c26c1ef2d961bc61d6ebb5b52db51510504e24377a40bd2c69675474d2a79bb9"
    },
    "resource_sha256": {
        "visualization/style.toml": "5dcaf9f879cbe9629cc2670f7d6a5f1ab80f3271a032624274fa05210043f677",
        "visualization/registry.toml": "9872b154498a31eb0fd545cd62127d3f01ebd98ebb8e5269be91f674f51378e3"
    },
    "definition_sha256": {
        "objectives.py": "22358a00ad9317e70b363e119e72201f2138a67430596b79ecd3aa3b9856b3a0",
        "evaluation/anchor_sampling.py": "050b1fb7dcb64709e279709bda45bf687a0fc51870ed982efdccc291903e082c",
        "evaluation/anchor_q_reporting.py": "2d04e4b047737babc151fff2f331079b6f486cf64f2886dd058fd988f7634c8f",
        "evaluation/interpretation.py": "3d5631d4df0d76c6234a487d0a6bd0a80dd4932ee3d3be338072e76b05139f41",
        "research/artifacts.py": "4d17b9f045852b756ea23a2312aa2de35cf712d004151819fe852710e3538f86",
        "evaluation/evaluation_outputs.py": "37e4db596d6aa468aedbb6eb6f7743f9946b68528d0da501a9c13a66912b02f1",
        "evaluation/interpretation_outputs.py": "2d1421aa6cb59df48d71f6c5a1f7f270ed3c6fe5125fda00dcf8133be67caec5",
        "evaluation/interpretation_inputs.py": "18fe6c14d0813aa5710a003564df084873618893cecc9041dcf6cbd2d6bcbc6a",
        "evaluation/interpretation_reporting.py": "b17d7f43bedadf5b6a6d135f340f37f0859f3019c22358ec2fb6bce16cf6f5c1",
        "evaluation/evaluation_tables.py": "6d7a25ed47b4c64b239657a6b60e415cbf50f8f9445e9bfa47ca37770f8591dc",
        "visualization/core.py": "0f779bfbf0459226ecbe068f6e4cac3c66d5f55d5497c41fb1a018723d50186b",
        "visualization/interpretation.py": "aa4b3765d8bc6d5e41aa54e376b032381dfdc75a3c057a2f38d7554018af36d3",
        "visualization/evaluation.py": "59ac756b7d86ca1091abe45c15f8d09308db83462c6be427c165a591f38e31ef",
        "visualization/interpretation_report.py": "9c985e852a17abb0bf9f86ff67cd5da8295678a79134c6fba7e538845cf337d7",
        "visualization/evaluation_report.py": "fc2d9c82683109cc3ad2190136995f1df287dbb9394a9b4297b8d61e8ce4fb08",
        "visualization/evaluation_workflow.py": "d0e030efdd6eb94b62523b70f1e6be59219ea1a8322022073076c6acfc6572ba"
    },
    "generator_sha256": "64f35b19a36808fb535f4c03f3eed069ea0678bb1a81170f7fffcd60ff6e6bbd"
}


## 3. Arial and figure settings

Use installed Arial or provide font files in the parameter cell. Font substitution and automatic font downloads are disabled.


In [ ]:
for filename in FONT_FILES:
    filename = Path(filename).resolve(strict=True)
    font_manager.fontManager.addfont(str(filename))
if not any(font.name == "Arial" for font in font_manager.fontManager.ttflist):
    raise RuntimeError("Arial is unavailable. Set FONT_FILES to existing Arial regular and bold files, or run in an environment with Arial installed.")

_style_resources = tempfile.TemporaryDirectory(prefix="evaluation-style-")
STYLE_RESOURCE_ROOT = Path(_style_resources.name)


In [ ]:
STYLE_TOML = r'''[schema]
version = 1

[font]
family = "Arial"
axis_label_pt = 9.0
tick_pt = 8.0
legend_pt = 8.0
panel_heading_pt = 9.0
panel_letter_pt = 10.0
annotation_pt = 8.0
normal_weight = "normal"
panel_letter_weight = "bold"

[color]
background = "#FFFFFF"
foreground = "#171717"
secondary = "#666666"
tertiary = "#888888"
quaternary = "#999999"

[line]
data_pt = 1.2
axes_pt = 0.7
reference_pt = 0.7
interval_pt = 0.9
caps_pt = 3.0

[marker]
size_pt = 4.0
edge_pt = 0.8
face = "white"
markevery_target = 10

[layout]
single_width_mm = 90.0
multi_width_mm = 180.0
panel_aspect = 1.3333333333333333
left_mm = 14.0
right_mm = 5.0
bottom_mm = 12.0
top_mm = 8.0
hgap_mm = 14.0
vgap_mm = 16.0
legend_row_mm = 8.0
legend_gap_mm = 3.0
default_columns = [1, 2, 2, 2, 3, 3, 3]

[axes]
grid = false
minor_ticks = false
tick_direction = "in"
major_length_pt = 3.0
major_width_pt = 0.7
major_pad_pt = 3.0
major_target = 5
panel_label_x_mm = -3.0
panel_label_y_mm = 2.0

[legend]
max_per_row = 4
handle_length_em = 2.4
column_spacing_em = 1.2
location = "upper center"

[histogram]
density = true
fill = false
bootstrap_bins = 40

[export]
formats = ["png", "eps"]
dpi = 300
transparent = false
overwrite = false

[backend.rcparams]
"pdf.fonttype" = 42
"ps.fonttype" = 42
"svg.fonttype" = "none"
'''
_ = (STYLE_RESOURCE_ROOT / 'style.toml').write_text(STYLE_TOML, encoding='utf-8')

REGISTRY_TOML = r'''[schema]
version = 1

[figures.pilot_ecdf]
status = "approved"
kind = "ecdf"
name = "Cosine distributions"
xlabel = "Cosine similarity"
ylabel = "ECDF"
series = ["base", "zh_cn"]
order = ["base", "zh_cn"]
scale = "linear"
xlim = [-1.0, 1.0]
ylim = [0.0, 1.0]

[figures.pilot_histogram]
status = "approved"
kind = "histogram"
name = "Cosine density"
xlabel = "Cosine similarity"
ylabel = "Density"
series = ["base", "zh_cn"]
order = ["base", "zh_cn"]
scale = "linear"
density = true
xlim = [-1.0, 1.0]
ylim = [0.0]
edges_start = -1.0
edges_stop = 1.0
edges_count = 61

[figures.pilot_bootstrap]
status = "approved"
kind = "bootstrap_histogram"
name = "Bootstrap differences"
xlabel = "Zh-cn \u2212 Base"
ylabel = "Bootstrap repeats"
series = ["delta"]
order = ["delta"]
scale = "linear"
bins = 40
density = false
ylim = [0.0]

[semantics.base]
label = "Base"
color = "#171717"
linestyle = "solid"
marker = "o"

[semantics.zh_cn]
label = "Zh-cn"
color = "#666666"
linestyle = "dashed"
marker = "s"

[semantics.delta]
label = "Zh-cn \u2212 Base"
color = "#171717"
linestyle = "solid"
marker = "o"

[figures.dimsim_distributions]
status = "approved"
kind = "distribution_triptych"
name = "DIMSIM distributions"
xlabel = "DIMSIM distance"
ylabel = "Pair count / ECDF"
series = ["population"]
order = ["population"]
scale = "linear"

[figures.dimsim_overlap_distributions]
status = "approved"
kind = "ecdf"
name = "Component overlap distributions"
xlabel = "DIMSIM distance"
ylabel = "ECDF"
series = ["000", "001", "010", "011", "100", "101", "110", "111"]
order = ["000", "001", "010", "011", "100", "101", "110", "111"]
scale = "linear"
ylim = [0.0, 1.0]

[figures.dimsim_component_contributions]
status = "approved"
kind = "stacked_bar"
name = "DIMSIM component contributions"
xlabel = "DIMSIM interval"
ylabel = "Share of mean DIMSIM distance"
series = ["initial", "final", "tone"]
order = ["initial", "final", "tone"]
scale = "linear"

[semantics.population]
label = "All pairs"
color = "#171717"
linestyle = "solid"
marker = "o"

[semantics.000]
label = "000"
color = "#171717"
linestyle = "solid"
marker = "o"
[semantics.001]
label = "001"
color = "#444444"
linestyle = "dashed"
marker = "s"
[semantics.010]
label = "010"
color = "#666666"
linestyle = "dotted"
marker = "^"
[semantics.011]
label = "011"
color = "#888888"
linestyle = "dashdot"
marker = "v"
[semantics.100]
label = "100"
color = "#303030"
linestyle = "solid"
marker = "D"
[semantics.101]
label = "101"
color = "#606060"
linestyle = "dashed"
marker = "P"
[semantics.110]
label = "110"
color = "#909090"
linestyle = "dotted"
marker = "X"
[semantics.111]
label = "111"
color = "#c0c0c0"
linestyle = "dashdot"
marker = "*"
[semantics.initial]
label = "Initial"
color = "#303030"
linestyle = "solid"
marker = "o"
[semantics.final]
label = "Final"
color = "#888888"
linestyle = "solid"
marker = "s"
[semantics.tone]
label = "Tone"
color = "#d0d0d0"
linestyle = "solid"
marker = "^"

[figures.anchor_q_transitions]
status = "approved"
kind = "dotplot"
name = "Anchor-Q transitions"
xlabel = "Q"
ylabel = "Transition"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
scale = "linear"

[figures.transition_shares]
status = "approved"
kind = "stacked_bar"
name = "Transition shares"
xlabel = "Transition"
ylabel = "Q share"
series = ["initial", "final", "tone"]
order = ["initial", "final", "tone"]
scale = "linear"

[figures.component_shares]
status = "approved"
kind = "stacked_bar"
name = "Component shares"
xlabel = "Model"
ylabel = "Q share"
series = ["initial", "final", "tone"]
order = ["initial", "final", "tone"]
scale = "linear"

[figures.dimsim_geometry]
status = "approved"
kind = "line_histogram"
name = "DIMSIM geometry"
xlabel = "DIMSIM distance"
ylabel = "Cosine similarity"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
scale = "linear"

[figures.dimsim_correlation]
status = "approved"
kind = "dotplot"
name = "DIMSIM correlation"
xlabel = "Model"
ylabel = "Spearman rho"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
scale = "linear"

[semantics.syllable]
label = "Syllable"
color = "#171717"
linestyle = "solid"
marker = "o"
[semantics.components]
label = "Components"
color = "#444444"
linestyle = "dashed"
marker = "s"
[semantics.acoustic]
label = "Acoustic"
color = "#707070"
linestyle = "dotted"
marker = "^"
[semantics."components-acoustic-lambda-1"]
label = "Combined \u03bb1"
color = "#999999"
linestyle = "dashdot"
marker = "D"
[semantics."components-acoustic-lambda-10"]
label = "Combined \u03bb10"
color = "#c5c5c5"
linestyle = "solid"
marker = "v"

[figures.residual_qq]
status = "approved"
kind = "dotplot"
name = "Residual Q\u2013Q"
xlabel = "Normal quantile"
ylabel = "Standardized residual"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10", "dimsim"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10", "dimsim"]
columns = 2
scale = "linear"

[semantics.dimsim]
label = "DIMSIM"
color = "#171717"
linestyle = "solid"
marker = "o"

[figures.training_objectives]
status = "approved"
kind = "line_histogram"
name = "Training objectives"
xlabel = "Epoch"
ylabel = "Objective"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
columns = 3
scale = "linear"

[figures.validation_accuracy]
status = "approved"
kind = "line_histogram"
name = "Validation accuracy"
xlabel = "Epoch"
ylabel = "Accuracy"
series = ["syllable", "components", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
columns = 2
scale = "linear"

[figures.validation_loss_components]
status = "approved"
kind = "line_histogram"
name = "Validation loss components"
xlabel = "Epoch"
ylabel = "Loss"
series = ["components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
columns = 2
scale = "linear"

[figures.component_accuracy]
status = "approved"
kind = "line_histogram"
name = "Component accuracy"
xlabel = "Epoch"
ylabel = "Accuracy"
series = ["initial", "final", "tone"]
order = ["initial", "final", "tone"]
columns = 2
scale = "linear"

[figures.invalid_if_predictions]
status = "approved"
kind = "line_histogram"
name = "Invalid initial\u2013final combinations"
xlabel = "Epoch"
ylabel = "Invalid predictions (%)"
series = ["components"]
order = ["components"]
columns = 1
scale = "linear"
'''
_ = (STYLE_RESOURCE_ROOT / 'registry.toml').write_text(REGISTRY_TOML, encoding='utf-8')



### Model identities


In [ ]:
BASE_MODEL_IDS = ('syllable', 'components', 'acoustic')
COMBINED_PREFIX = 'components-acoustic-lambda-'

def combined_model_id(weight: Real) -> str:
    """Encode a finite coefficient without truncating its float precision."""
    if isinstance(weight, bool) or not isinstance(weight, Real):
        raise ValueError('lambda must be a finite real number')
    try:
        value = float(weight)
    except (OverflowError, ValueError) as exc:
        raise ValueError('lambda must be a finite real number') from exc
    if not math.isfinite(value):
        raise ValueError('lambda must be a finite real number')
    token = str(int(value)) if value.is_integer() and abs(value) < 1e+16 else repr(value).replace('e+', 'e')
    return COMBINED_PREFIX + token

def parse_model_id(model_id: str) -> tuple[str, float | None]:
    """Return the objective family and the combined acoustic coefficient."""
    if isinstance(model_id, str) and model_id in BASE_MODEL_IDS:
        return (model_id, None)
    if not isinstance(model_id, str) or not model_id.startswith(COMBINED_PREFIX):
        raise ValueError('unknown objective model_id')
    token = model_id[len(COMBINED_PREFIX):]
    if not re.fullmatch('[+-]?(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:[eE][+-]?\\d+)?', token):
        raise ValueError('combined model_id must contain a finite numeric lambda')
    weight = float(token)
    if not math.isfinite(weight):
        raise ValueError('lambda must be a finite real number')
    return ('components-acoustic', weight)

def canonical_model_id(model_id: str) -> str:
    family, weight = parse_model_id(model_id)
    return combined_model_id(weight) if family == 'components-acoustic' else family


### Fixed transition definitions


In [ ]:
EDGES = (('initial', '000', '100'), ('initial', '001', '101'), ('initial', '010', '110'), ('initial', '011', '111'), ('final', '000', '010'), ('final', '001', '011'), ('final', '100', '110'), ('final', '101', '111'), ('tone', '000', '001'), ('tone', '010', '011'), ('tone', '100', '101'), ('tone', '110', '111'))


### Paired Anchor-Q summaries


In [ ]:
SEEDS = (0, 1, 2, 3, 4)
COMPONENTS = ('initial', 'final', 'tone')
METRICS = (*COMPONENTS, 'overall')
EDGE_ORDER = tuple((f'{component}:{low}->{high}' for component, low, high in EDGES))

def q_is_number(value: Any) -> bool:
    return isinstance(value, Real) and (not isinstance(value, bool)) and math.isfinite(float(value))

def _sample_sd(values: list[float]) -> float:
    return float(np.std(np.asarray(values, dtype=np.float64), ddof=1))

def _mean_se(draws: list[np.ndarray]) -> tuple[float, float | None]:
    """MC SE of the seed-mean estimator, retaining corresponding draw pairing."""
    pooled = np.mean(np.stack(draws, axis=0), axis=0)
    if pooled.size < 2:
        return (float(np.mean(pooled)), None)
    return (float(np.mean(pooled)), float(np.std(pooled, ddof=1) / math.sqrt(pooled.size)))

def _edge_ids(draws: list[dict[str, Any]], edges: list[str]) -> tuple[list[tuple[str, str, str]], dict[str, list[float]]]:
    identities = []
    values = {edge: [] for edge in edges}
    for row in draws:
        if not isinstance(row, dict):
            raise ValueError('draw score rows must be objects')
        draw_id, edge_id, triplet_id = (row.get(key) for key in ('draw_id', 'edge_id', 'triplet_id'))
        if not all((isinstance(x, str) and x for x in (draw_id, edge_id, triplet_id))):
            raise ValueError('draw identities must be non-empty strings')
        if edge_id not in values:
            raise ValueError(f'unexpected edge identity: {edge_id}')
        score = row.get('score')
        if not q_is_number(score) or float(score) not in (0.0, 0.5, 1.0):
            raise ValueError('draw score must be numeric 0, 0.5, or 1')
        identities.append((draw_id, edge_id, triplet_id))
        values[edge_id].append(float(score))
    if len({item[0] for item in identities}) != len(identities):
        raise ValueError('draw IDs must be unique within each seed')
    expected_edge_sequence = [edge for edge in edges for _ in values[edge]]
    if [item[1] for item in identities] != expected_edge_sequence:
        raise ValueError('draw rows must follow canonical edge blocks')
    return (identities, values)

def build_anchor_q_report(batch_result: dict[str, Any], reference_model: str='syllable') -> dict[str, Any]:
    """Validate paired five-seed batch scores and return tidy JSON-compatible data.

    ``tables`` contains per-model/per-seed edges and metrics, per-model seed
    summaries, per-model global summaries, paired deltas, and normalized shares.
    ``plot_data`` contains edge means with their five seed points and the two share
    series. All values retain full numeric precision. Missing one-draw MC SE is
    represented by ``None`` and an explicit status.
    """
    if not isinstance(batch_result, dict) or type(batch_result.get('schema_version')) is not int or batch_result['schema_version'] != 1:
        raise ValueError('batch schema_version must be integer 1')
    bundle_hash = batch_result.get('bundle_manifest_sha256')
    if not isinstance(bundle_hash, str) or len(bundle_hash) != 64 or any((c not in '0123456789abcdef' for c in bundle_hash)):
        raise ValueError('invalid batch bundle manifest SHA-256')
    try:
        reference = canonical_model_id(reference_model)
    except (TypeError, ValueError) as error:
        raise ValueError('invalid reference model') from error
    records = batch_result.get('results')
    if not isinstance(records, list) or not records:
        raise ValueError('batch results are required')
    models = set()
    for record in records:
        if not isinstance(record, dict) or not isinstance(record.get('model_id'), str):
            raise ValueError('each batch record requires a model ID')
        try:
            model = canonical_model_id(record['model_id'])
        except (TypeError, ValueError) as error:
            raise ValueError('invalid model ID') from error
        if model != record['model_id']:
            raise ValueError('model IDs must be canonical')
        models.add(model)
    if reference not in models:
        raise ValueError('reference model is absent from batch')
    model_order = sorted(models)
    grouped: dict[str, dict[int, dict[str, Any]]] = {model: {} for model in model_order}
    for record in records:
        model, seed = (record['model_id'], record.get('seed'))
        if type(seed) is not int or seed not in SEEDS or seed in grouped[model]:
            raise ValueError(f'invalid or duplicate seed for {model}')
        score = record.get('score')
        if not isinstance(score, dict):
            raise ValueError('score must be an object')
        if score.get('model_id') != model or type(score.get('seed')) is not int or score['seed'] != seed:
            raise ValueError('internal model/seed metadata mismatch')
        if score.get('bundle_manifest_sha256') != bundle_hash:
            raise ValueError('bundle hash mismatch')
        provenance = score.get('provenance')
        if not isinstance(provenance, dict) or provenance.get('model_id') != model or type(provenance.get('seed')) is not int or (provenance['seed'] != seed) or (provenance.get('bundle_manifest_sha256') != bundle_hash):
            raise ValueError('score provenance model/seed/bundle metadata mismatch')
        tolerance = score.get('tie_tolerance')
        if not q_is_number(tolerance) or tolerance < 0:
            raise ValueError('invalid tie tolerance')
        grouped[model][seed] = score
    if any((set(scores) != set(SEEDS) for scores in grouped.values())):
        raise ValueError('every model requires seeds 0 through 4 exactly once')
    first_score = grouped[model_order[0]][SEEDS[0]]
    tolerance = float(first_score['tie_tolerance'])
    edge_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    all_draw_values: dict[str, dict[int, dict[str, np.ndarray]]] = {model: {} for model in model_order}
    identity_reference = None
    draw_count = None
    coverage_reference = None
    for model in model_order:
        for seed in SEEDS:
            score = grouped[model][seed]
            if float(score['tie_tolerance']) != tolerance:
                raise ValueError('tie tolerance mismatch')
            edge_summary = score.get('edges')
            if not isinstance(edge_summary, dict) or set(edge_summary) != set(EDGE_ORDER):
                raise ValueError('score edges must contain the canonical twelve-edge inventory')
            draws = score.get('draw_scores')
            if not isinstance(draws, list) or not draws:
                raise ValueError('draw_scores must be a non-empty list')
            identities, edge_values = _edge_ids(draws, list(EDGE_ORDER))
            expected_draw_ids = [f'anchor-q:v1:{edge}:{index:06d}' for edge in EDGE_ORDER for index in range(len(edge_values[edge]))]
            if [identity[0] for identity in identities] != expected_draw_ids:
                raise ValueError('draw IDs do not match canonical edge/index order')
            if identity_reference is None:
                identity_reference = identities
            elif identities != identity_reference:
                raise ValueError('draw identities differ across models or seeds')
            sizes = {edge: len(edge_values[edge]) for edge in EDGE_ORDER}
            if len(set(sizes.values())) != 1:
                raise ValueError('all edge draw counts must match')
            if draw_count is None:
                draw_count = next(iter(sizes.values()))
            if any((count != draw_count for count in sizes.values())):
                raise ValueError('edge draw counts differ across models or seeds')
            coverage = score.get('coverage')
            if not isinstance(coverage, dict) or not isinstance(coverage.get('edges'), dict) or set(coverage['edges']) != set(EDGE_ORDER):
                raise ValueError('per-edge coverage diagnostics are required')
            if coverage_reference is None:
                coverage_reference = coverage
            elif coverage != coverage_reference:
                raise ValueError('coverage diagnostics differ across models or seeds')
            arrays = {edge: np.asarray(edge_values[edge], dtype=np.float64) for edge in EDGE_ORDER}
            all_draw_values[model][seed] = arrays
            q = {}
            mcse = {}
            for edge in EDGE_ORDER:
                vals = arrays[edge]
                q[edge] = float(np.mean(vals))
                mcse[edge] = float(np.std(vals, ddof=1) / math.sqrt(vals.size)) if vals.size > 1 else None
                declared = edge_summary[edge]
                if not isinstance(declared, dict) or type(declared.get('draw_count')) is not int or declared['draw_count'] != vals.size:
                    raise ValueError('declared edge draw_count differs from actual draws')
                cov = coverage['edges'][edge]
                edge_rows.append({'model_id': model, 'seed': seed, 'edge_id': edge, 'q': q[edge], 'mc_se': mcse[edge], 'mc_se_status': 'ok' if mcse[edge] is not None else 'not_available_single_draw', 'draw_count': int(vals.size), 'coverage': cov})
            for component in COMPONENTS:
                edges = [edge for edge in EDGE_ORDER if edge.startswith(component + ':')]
                metric_rows.append({'model_id': model, 'seed': seed, 'metric': component, 'q': float(np.mean([q[e] for e in edges])), 'mc_se': float(math.sqrt(sum((mcse[e] ** 2 for e in edges))) / len(edges)) if all((mcse[e] is not None for e in edges)) else None, 'mc_se_status': 'ok' if all((mcse[e] is not None for e in edges)) else 'not_available_single_draw'})
            metric_rows.append({'model_id': model, 'seed': seed, 'metric': 'overall', 'q': float(np.mean([q[e] for e in EDGE_ORDER])), 'mc_se': float(math.sqrt(sum((mcse[e] ** 2 for e in EDGE_ORDER))) / len(EDGE_ORDER)) if all((mcse[e] is not None for e in EDGE_ORDER)) else None, 'mc_se_status': 'ok' if all((mcse[e] is not None for e in EDGE_ORDER)) else 'not_available_single_draw'})
    edge_summary_rows = []
    for model in model_order:
        for edge in EDGE_ORDER:
            seed_values = [float(np.mean(all_draw_values[model][seed][edge])) for seed in SEEDS]
            mean_q, mean_mcse = _mean_se([all_draw_values[model][seed][edge] for seed in SEEDS])
            edge_summary_rows.append({'model_id': model, 'edge_id': edge, 'mean': mean_q, 'sample_sd': _sample_sd(seed_values), 'mc_se': mean_mcse, 'mc_se_status': 'ok' if mean_mcse is not None else 'not_available_single_draw', 'seed_values': [{'seed': seed, 'q': val} for seed, val in zip(SEEDS, seed_values)]})
    metric_summary_rows = []
    global_summary_rows = []
    for model in model_order:
        for metric in METRICS:
            edge_subset = list(EDGE_ORDER) if metric == 'overall' else [e for e in EDGE_ORDER if e.startswith(metric + ':')]
            seed_values = []
            edge_mean_ses = []
            for seed in SEEDS:
                edge_means = [float(np.mean(all_draw_values[model][seed][edge])) for edge in edge_subset]
                seed_values.append(float(np.mean(edge_means)))
            mean_q = float(np.mean(seed_values))
            for edge in edge_subset:
                _, edge_mean_se = _mean_se([all_draw_values[model][seed][edge] for seed in SEEDS])
                edge_mean_ses.append(edge_mean_se)
            mean_mcse = float(math.sqrt(sum((value ** 2 for value in edge_mean_ses))) / len(edge_subset)) if all((value is not None for value in edge_mean_ses)) else None
            row = {'model_id': model, 'metric': metric, 'mean': mean_q, 'sample_sd': _sample_sd(seed_values), 'mc_se': mean_mcse, 'mc_se_status': 'ok' if mean_mcse is not None else 'not_available_single_draw', 'seed_values': [{'seed': seed, 'q': val} for seed, val in zip(SEEDS, seed_values)]}
            metric_summary_rows.append(row)
            global_summary_rows.append(dict(row))
    paired_rows = []
    for model in model_order:
        if model == reference:
            continue
        for metric in METRICS:
            edge_subset = list(EDGE_ORDER) if metric == 'overall' else [e for e in EDGE_ORDER if e.startswith(metric + ':')]
            seed_deltas = []
            mean_draw_differences = []
            for seed in SEEDS:
                edge_diff_means = []
                for edge in edge_subset:
                    diffs = all_draw_values[model][seed][edge] - all_draw_values[reference][seed][edge]
                    edge_diff_means.append(float(np.mean(diffs)))
                seed_deltas.append(float(np.mean(edge_diff_means)))
            for edge in edge_subset:
                across_seeds = np.mean(np.stack([all_draw_values[model][seed][edge] - all_draw_values[reference][seed][edge] for seed in SEEDS]), axis=0)
                mean_draw_differences.append(across_seeds)
            edge_mcse = [float(np.std(values, ddof=1) / math.sqrt(values.size)) if values.size > 1 else None for values in mean_draw_differences]
            paired_mcse = float(math.sqrt(sum((value ** 2 for value in edge_mcse)) / len(edge_mcse) ** 2)) if all((value is not None for value in edge_mcse)) else None
            paired_rows.append({'model_id': model, 'reference_model': reference, 'metric': metric, 'mean_delta_q': float(np.mean(seed_deltas)), 'sample_sd_seed_delta': _sample_sd(seed_deltas), 'mc_se': paired_mcse, 'mc_se_status': 'ok' if paired_mcse is not None else 'not_available_single_draw', 'seed_deltas': [{'seed': seed, 'delta_q': value} for seed, value in zip(SEEDS, seed_deltas)]})
    share_rows = []
    component_share_rows = []
    for model in model_order:
        means = {row['edge_id']: row['mean'] for row in edge_summary_rows if row['model_id'] == model}
        total = sum(means.values())
        for edge in EDGE_ORDER:
            share_rows.append({'model_id': model, 'edge_id': edge, 'share': means[edge] / total if total else None, 'status': 'ok' if total else 'undefined_zero_total'})
        comp_means = {component: float(np.mean([means[e] for e in EDGE_ORDER if e.startswith(component + ':')])) for component in COMPONENTS}
        comp_total = sum(comp_means.values())
        for component in COMPONENTS:
            component_share_rows.append({'model_id': model, 'component': component, 'share': comp_means[component] / comp_total if comp_total else None, 'status': 'ok' if comp_total else 'undefined_zero_total'})
    plot_edges = [{'model_id': model, 'edge_id': edge, 'mean': next((row['mean'] for row in edge_summary_rows if row['model_id'] == model and row['edge_id'] == edge)), 'seed_points': [{'seed': seed, 'q': float(np.mean(all_draw_values[model][seed][edge]))} for seed in SEEDS]} for model in model_order for edge in EDGE_ORDER]
    return {'schema_version': 1, 'contract': {'seeds': list(SEEDS), 'edge_order': list(EDGE_ORDER), 'metrics': list(METRICS), 'reference_model': reference, 'bundle_manifest_sha256': bundle_hash, 'tie_tolerance': tolerance, 'draws_per_edge': draw_count, 'full_precision': True, 'seed_summary': 'mean and sample SD (ddof=1) across five seeds', 'mc_uncertainty': 'MC SE from draw scores averaged across seeds before sample SD / sqrt(draw count); independent-edge aggregation', 'paired_uncertainty': 'paired draw differences, averaged across seeds before MC SE; no confidence intervals', 'tables': ['model_seed_edges', 'model_seed_metrics', 'model_edge_seed_summary', 'model_metric_seed_summary', 'global_metric_summary', 'paired_deltas', 'transition_shares', 'component_shares'], 'plot_data': ['edges with mean and five seed points', 'transition shares', 'component shares']}, 'tables': {'model_seed_edges': edge_rows, 'model_seed_metrics': metric_rows, 'model_edge_seed_summary': edge_summary_rows, 'model_metric_seed_summary': metric_summary_rows, 'global_metric_summary': global_summary_rows, 'paired_deltas': paired_rows, 'transition_shares': share_rows, 'component_shares': component_share_rows}, 'plot_data': {'edges': plot_edges, 'transition_shares': share_rows, 'component_shares': component_share_rows}}


### Categorical fits and residual tables


In [ ]:
MODELS = ('syllable', 'components', 'acoustic', 'components-acoustic-lambda-1', 'components-acoustic-lambda-10')
SEEDS = tuple(range(5))
MASKS = tuple((f'{i:03b}' for i in range(8)))
LABELS = dict(zip(MODELS, ('Syllable', 'Components', 'Acoustic', 'Combined λ1', 'Combined λ10')))
INTERPRETATION_SETTINGS = {'sample': 'dimsim-balanced', 'mask_order': ['initial', 'final', 'tone'], 'masks': list(MASKS), 'models': list(MODELS), 'seeds': list(SEEDS), 'learned_outcome': 'cosine_similarity', 'reference_outcome': '-dimsim_distance', 'fit': 'eight categorical means; no separate intercept', 'r2': 'centered', 'draw_weights': 'original multiplicity', 'seed_summary': 'mean and sample SD', 'top_limit': 50, 'top_ranking': 'positive raw residual; learned mean across seeds', 'top_deduplication': 'unordered occurrence pair', 'inference': False, 'bootstrap': False, 'p_values': False}

def residual_finite_number(value, name):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or (not math.isfinite(value)):
        raise ValueError(f'{name} must be a finite number')
    return float(value)

def categorical_fit(outcome, masks):
    """Return category means, raw residuals and centered descriptive R²."""
    y = np.asarray([residual_finite_number(v, 'outcome') for v in outcome], dtype=np.float64)
    masks = list(masks)
    if not len(y) or len(y) != len(masks) or any((m not in MASKS for m in masks)):
        raise ValueError('nonempty aligned outcomes and binary I/F/T masks are required')
    beta, counts, diagnostics = ({}, {}, [])
    residual = np.empty_like(y)
    for mask in MASKS:
        idx = np.array([m == mask for m in masks])
        counts[mask] = int(idx.sum())
        beta[mask] = float(y[idx].mean()) if counts[mask] else None
        if counts[mask]:
            residual[idx] = y[idx] - beta[mask]
            if np.ptp(y[idx]) == 0:
                diagnostics.append({'mask': mask, 'reason': 'constant_within_mask'})
        else:
            diagnostics.append({'mask': mask, 'reason': 'empty_category'})
    sst = float(np.sum((y - y.mean()) ** 2))
    sse = float(np.sum(residual ** 2))
    if sst == 0:
        diagnostics.append({'reason': 'constant_outcome'})
    return {'beta': beta, 'counts': counts, 'r2': 1 - sse / sst if sst else None, 'r2_status': 'ok' if sst else 'constant_outcome', 'sse': sse, 'sst': sst, 'residuals': residual.tolist(), 'diagnostics': diagnostics}

def residual_spearman(left, right):
    """Average tied ranks; undefined constants remain null, without p-values."""
    a, b = (np.asarray([residual_finite_number(v, 'residual') for v in values]) for values in (left, right))
    if len(a) != len(b) or not len(a):
        raise ValueError('nonempty aligned residual vectors are required')
    reason = 'fewer_than_two_draws' if len(a) < 2 else 'constant_learned_residual' if np.ptp(a) == 0 else 'constant_reference_residual' if np.ptp(b) == 0 else None
    if reason:
        return {'rho': None, 'status': reason}

    def ranks(x):
        _, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
        return (np.cumsum(counts) - (counts - 1) / 2)[inv]
    return {'rho': float(np.corrcoef(ranks(a), ranks(b))[0, 1]), 'status': 'ok'}

def residual_seed_summary(values):
    available = [v for v in values if v is not None]
    complete = len(available) == len(SEEDS)
    return {'values': values, 'mean': float(np.mean(available)) if complete else None, 'sample_sd': float(np.std(available, ddof=1)) if complete else None, 'defined_seeds': len(available), 'status': 'ok' if complete else 'undefined_seed_values'}

def required_identity_text(value, name):
    if not isinstance(value, str) or not value:
        raise ValueError(f'{name} must be a nonempty string')
    return value

def build_interpretation(records: Iterable[dict], occurrences: list[dict], sentences: list[dict], *, bundle_manifest_sha256: str) -> dict:
    """Consume saved DIMSIM result records one at a time; retain compact arrays.

    The caller supplies hash-verified input records and bundle metadata. This pure
    API validates their identities and provenance but does not verify file bytes.
    """
    if not isinstance(bundle_manifest_sha256, str) or len(bundle_manifest_sha256) != 64 or any((c not in '0123456789abcdef' for c in bundle_manifest_sha256)):
        raise ValueError('invalid bundle manifest SHA-256')
    occ = {r['occurrence_id']: r for r in occurrences}
    contexts = {r['sentence_id']: r for r in sentences}
    if len(occ) != len(occurrences) or len(contexts) != len(sentences):
        raise ValueError('duplicate occurrence or sentence IDs')
    fits, provenance, canonical, metadata, reference = ({}, {}, None, None, None)
    pair_identity_map = {}
    for record in records:
        model, seed = (record.get('model_id'), record.get('seed'))
        if model not in MODELS or type(seed) is not int or seed not in SEEDS or ((model, seed) in fits):
            raise ValueError('invalid or duplicate model/seed')
        score = record['score']
        if score.get('model_id') != model or score.get('seed') != seed or type(score.get('seed')) is not int or (score.get('bundle_manifest_sha256') != bundle_manifest_sha256):
            raise ValueError('score identity or bundle mismatch')
        pin = {k: required_identity_text(record.get(k), k) for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256')}
        for field in ('run_manifest_sha256', 'checkpoint_sha256'):
            if len(pin[field]) != 64 or any((c not in '0123456789abcdef' for c in pin[field])):
                raise ValueError('invalid provenance hash')
        if model in provenance and any((pin[k] != provenance[model][0][k] for k in ('run_id', 'run_manifest_sha256'))):
            raise ValueError('mixed training runs')
        provenance.setdefault(model, []).append({'seed': seed, **pin})
        rows = score['modes']['dimsim-balanced']['rows']
        if not rows:
            raise ValueError('balanced rows are required')
        ids, values, distances, repeated_scores = ([], [], [], {})
        for row in rows:
            if row.get('analysis_status') != 'ok' or row.get('distance_status') != 'ok':
                raise ValueError('invalid balanced draw status; no silent filtering')
            mask = row.get('overlap_mask')
            if mask not in MASKS:
                raise ValueError('invalid overlap mask')
            identity = tuple((required_identity_text(row.get(k), k) for k in ('draw_id', 'pair_id', 'left_id', 'right_id')))
            if identity[2] == identity[3]:
                raise ValueError('self-comparison is not permitted')
            distance = residual_finite_number(row.get('dimsim_distance'), 'DIMSIM distance')
            cosine = residual_finite_number(row.get('cosine_similarity'), 'cosine similarity')
            if distance < 0 or not -1 <= cosine <= 1:
                raise ValueError('outcome outside valid range')
            pair_key = tuple(sorted(identity[2:]))
            if pair_key in repeated_scores and repeated_scores[pair_key] != cosine:
                raise ValueError('repeated occurrence pair has inconsistent scores')
            repeated_scores[pair_key] = cosine
            ids.append((*identity, mask, distance))
            values.append(cosine)
            distances.append(distance)
        if len({r[0] for r in ids}) != len(ids):
            raise ValueError('duplicate draw IDs')
        if canonical is None:
            canonical = ids
            metadata = []
            reverse_pairs = {}
            for draw_id, pair_id, left, right, mask, distance in ids:
                unordered = tuple(sorted((left, right)))
                if pair_id in pair_identity_map and pair_identity_map[pair_id] != unordered:
                    raise ValueError('pair ID collision')
                if unordered in reverse_pairs and reverse_pairs[unordered] != (pair_id, mask, distance):
                    raise ValueError('inconsistent repeated occurrence pair')
                pair_identity_map[pair_id] = unordered
                reverse_pairs[unordered] = (pair_id, mask, distance)
                ends = []
                for oid in (left, right):
                    if oid not in occ:
                        raise ValueError('missing occurrence metadata')
                    o = occ[oid]
                    sentence = contexts.get(o['sample_id'])
                    if sentence is None:
                        raise ValueError('missing sentence context')
                    position = o['character_position']
                    if type(position) is not int or not 0 <= position < len(sentence['text']) or sentence['text'][position] != o['character']:
                        raise ValueError('occurrence/context alignment mismatch')
                    ends.append({'occurrence_id': oid, 'sentence_id': o['sample_id'], 'position': position, 'text': sentence['text'], 'character': o['character'], 'pinyin': o['pinyin']})
                actual_mask = ''.join(('1' if occ[left][k] == occ[right][k] else '0' for k in ('initial', 'final', 'tone')))
                if actual_mask != mask:
                    raise ValueError('mask disagrees with occurrence components')
                metadata.append({'draw_id': draw_id, 'pair_id': pair_id, 'mask': mask, 'left': ends[0], 'right': ends[1]})
            reference = categorical_fit([-d for d in distances], [r[4] for r in ids])
        elif ids != canonical:
            raise ValueError('balanced draw identities, order, masks or distances differ')
        fits[model, seed] = categorical_fit(values, [r[4] for r in ids])
    if set(fits) != {(m, s) for m in MODELS for s in SEEDS}:
        raise ValueError('exact five models by five seeds are required')
    summary, per_seed, residuals, tops = ([], [], {}, {})
    for model in (*MODELS, 'dimsim'):
        if model == 'dimsim':
            row = {'model': model, 'beta': reference['beta'], 'r2': reference['r2'], 'residual_spearman': None, 'seed_sd': None}
            raw = reference['residuals']
        else:
            selected = [fits[model, s] for s in SEEDS]
            correlations = [residual_spearman(f['residuals'], reference['residuals']) for f in selected]
            row = {'model': model, 'beta': {m: residual_seed_summary([f['beta'][m] for f in selected]) for m in MASKS}, 'r2': residual_seed_summary([f['r2'] for f in selected]), 'residual_spearman': residual_seed_summary([r['rho'] for r in correlations])}
            raw = np.mean([f['residuals'] for f in selected], axis=0).tolist()
            for seed, fit, corr in zip(SEEDS, selected, correlations):
                per_seed.append({'model': model, 'seed': seed, **{k: v for k, v in fit.items() if k != 'residuals'}, 'residual_spearman': corr})
            residuals[model] = {str(s): fits[model, s]['residuals'] for s in SEEDS}
        summary.append(row)
        seen, top = (set(), [])
        for index in sorted(range(len(raw)), key=lambda i: (-raw[i], metadata[i]['pair_id'], metadata[i]['draw_id'])):
            pair = metadata[index]
            identity = tuple(sorted((pair['left']['occurrence_id'], pair['right']['occurrence_id'])))
            if raw[index] <= 0 or identity in seen:
                continue
            seen.add(identity)
            if len(top) < 50:
                top.append({**pair, 'resid': raw[index]})
        tops[model] = {'rows': top, 'returned_count': len(top), 'positive_unique_pair_count': len(seen)}
    residuals['dimsim'] = reference['residuals']
    return {'schema_version': 1, 'config': copy.deepcopy(INTERPRETATION_SETTINGS), 'bundle_manifest_sha256': bundle_manifest_sha256, 'provenance': {m: sorted(v, key=lambda p: p['seed']) for m, v in provenance.items()}, 'summary': summary, 'per_seed': per_seed, 'reference_fit': reference, 'draw_metadata': metadata, 'residuals': residuals, 'top_pairs': tops, 'diagnostics': {'draw_count': len(canonical), 'unique_pairs': len(pair_identity_map), 'mask_counts': reference['counts'], 'observations_independent': False}}

def readable_tables(result):
    """Format the six-row coefficient table and six strictly six-column tables."""

    def fmt(value):
        if isinstance(value, dict):
            return '—' if value['mean'] is None else f"{value['mean']:.6f} ± {value['sample_sd']:.6f}"
        return '—' if value is None else f'{value:.6f}'
    lines = ['| Model | ' + ' | '.join(('β' + m for m in MASKS)) + ' | R² | Residual Spearman vs DIMSIM |', '| ' + ' | '.join(['---'] * 11) + ' |']
    for row in result['summary']:
        cells = [LABELS.get(row['model'], 'DIMSIM')] + [fmt(row['beta'][m]) for m in MASKS]
        cells += [fmt(row['r2']), fmt(row['residual_spearman'])]
        lines.append('| ' + ' | '.join(cells) + ' |')
    tables = {'coefficients.md': '\n'.join(lines) + '\n'}

    def escape(s):
        return str(s).replace('|', '\\|').replace('\n', ' ')
    for model, table in result['top_pairs'].items():
        lines = ['| char1 | char2 | pinyin1 | pinyin2 | mask | resid |', '| --- | --- | --- | --- | --- | --- |']
        for row in table['rows']:
            cells = [row['left']['character'], row['right']['character'], row['left']['pinyin'], row['right']['pinyin'], row['mask'], f"{row['resid']:.8f}"]
            lines.append('| ' + ' | '.join(map(escape, cells)) + ' |')
        tables[f'top-50-{model}.md'] = '\n'.join(lines) + f"\nReturned pairs: {table['returned_count']}.\n"
    return tables


### Input and output paths


In [ ]:
def file_sha256(path, chunk=1024 * 1024):
    h = sha256()
    with Path(path).open('rb') as f:
        while (data := f.read(chunk)):
            h.update(data)
    return h.hexdigest()

def safe_run_id(value):
    if not isinstance(value, str) or not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]{0,127}', value) or value in {'.', '..'}:
        raise ValueError('run_id must be a non-empty safe filename component')
    return value

def safe_analysis(value):
    if not isinstance(value, str) or not re.fullmatch('[a-z0-9][a-z0-9-]{0,63}', value):
        raise ValueError('analysis must be a safe lowercase name')
    return value

def ensure_output_outside_inputs(output, input_roots):
    output = Path(output).resolve()
    for root in map(lambda p: Path(p).resolve(), input_roots):
        if output == root or root in output.parents:
            raise ValueError(f'Output must not be inside input: {root}')


### Verified artifact loading and output manifests


In [ ]:
def data_json_bytes(value: Any) -> bytes:
    return (json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_data_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('xb') as stream:
        stream.write(data_json_bytes(value))

def artifact_files(root: Path) -> set[str]:
    return {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}

def read_finite_json(path: Path):

    def bad(token):
        raise ValueError(f'non-finite JSON constant: {token}')
    value = json.loads(path.read_text(encoding='utf-8'), parse_constant=bad)
    data_json_bytes(value)
    return value

def check_input_pins(inputs, scientific_pins):
    bundle_pin = scientific_pins.get('bundle_manifest_sha256')
    if bundle_pin and bundle_pin not in {x['sha256'] for x in inputs}:
        raise ValueError('scientific bundle manifest pin does not match an input identity')
    exp_pin = scientific_pins.get('experiment_manifest_sha256')
    if exp_pin and exp_pin not in {x['sha256'] for x in inputs}:
        raise ValueError('scientific experiment manifest pin does not match an input identity')

def verify_artifact_inventory(root: Path, analysis: str, run_id: str, expected: set[str], *, profile: str='numerical_outputs', figure_exports: str='pending'):
    safe_analysis(analysis)
    safe_run_id(run_id)
    manifest = read_finite_json(root / 'manifest.json')
    if manifest.get('schema_version') != 1 or manifest.get('status') != 'complete' or manifest.get('analysis') != analysis or (manifest.get('run_id') != run_id) or (manifest.get('scope') != profile) or (manifest.get('figure_exports') != figure_exports) or (set(manifest.get('required_artifacts', [])) != expected):
        raise ValueError('output manifest identity/profile mismatch')
    if not isinstance(manifest.get('python'), str) or not manifest['python'] or (not isinstance(manifest.get('dependencies'), dict)) or ('numpy' not in manifest['dependencies']) or (not isinstance(manifest.get('inputs'), list)) or (not isinstance(manifest.get('source_hashes'), dict)) or (not manifest['source_hashes']):
        raise ValueError('output provenance metadata is incomplete')
    for row in manifest['inputs']:
        if not isinstance(row, dict) or not isinstance(row.get('path'), str) or (not isinstance(row.get('identity_path'), str)) or (not is_sha256(row.get('sha256'))):
            raise ValueError('output input identity metadata is invalid')
    if any((not is_sha256(x) for x in manifest['source_hashes'].values())):
        raise ValueError('output source hash metadata is invalid')
    actual, inventory = (artifact_files(root) - {'manifest.json'}, manifest.get('artifacts'))
    declared = {x.get('path') for x in inventory if isinstance(x, dict)} if isinstance(inventory, list) else set()
    if actual != expected or declared != expected or len(inventory or []) != len(expected):
        raise ValueError('output artifact inventory mismatch')
    for row in inventory:
        if not isinstance(row, dict) or set(row) != {'path', 'sha256', 'bytes'}:
            raise ValueError('malformed artifact inventory row')
        path = root / row['path']
        if not path.is_file() or path.stat().st_size <= 0 or path.stat().st_size != row['bytes'] or (file_sha256(path) != row['sha256']):
            raise ValueError(f"artifact hash or size mismatch: {row.get('path')}")
    return manifest

def is_sha256(value):
    return isinstance(value, str) and len(value) == 64 and all((c in '0123456789abcdef' for c in value))

@contextmanager
def presentation_run(project_root: str | Path, analysis: str, run_id: str, *, input_roots: list[str | Path], config: dict, source_hashes: dict, expected_files: set[str], scientific_pins: dict, create_data: bool=True, output_path: str | Path | None=None, profile: str='numerical_outputs', figure_exports_on_complete: str='pending'):
    """Write one run, gating every file except its top-level manifest."""
    analysis, run_id = (safe_analysis(analysis), safe_run_id(run_id))
    if figure_exports_on_complete not in {'pending', 'complete'}:
        raise ValueError('unsupported figure export completion status')
    root = Path(output_path).resolve() if output_path is not None else Path(project_root).resolve() / 'outputs' / analysis / run_id
    ensure_output_outside_inputs(root, input_roots)
    if root.exists():
        raise FileExistsError(f'refusing to overwrite existing output directory: {root}')
    if not isinstance(config, dict) or not isinstance(source_hashes, dict) or (not source_hashes):
        raise ValueError('explicit config and source hash map are required')
    if any((not isinstance(v, str) or len(v) != 64 or any((c not in '0123456789abcdef' for c in v)) for v in source_hashes.values())):
        raise ValueError('source hashes must be lowercase SHA-256 digests')
    data_json_bytes({'config': config, 'source_hashes': source_hashes, 'scientific_pins': scientific_pins})
    inputs = []
    for raw in input_roots:
        p = Path(raw).resolve(strict=True)
        identity = p if p.is_file() else p / 'manifest.json'
        if not identity.is_file():
            raise FileNotFoundError(f'input manifest identity missing: {identity}')
        inputs.append({'path': str(p), 'identity_path': str(identity), 'sha256': file_sha256(identity)})
    check_input_pins(inputs, scientific_pins)
    dependencies = {}
    for package in ('numpy', 'scipy', 'pandas', 'matplotlib'):
        try:
            dependencies[package] = version(package)
        except PackageNotFoundError:
            dependencies[package] = None
    root.mkdir(parents=True, exist_ok=False)
    manifest = {'schema_version': 1, 'analysis': analysis, 'run_id': run_id, 'status': 'running', 'scope': profile, 'figure_exports': 'pending', 'required_artifacts': sorted(expected_files), 'artifacts': [], 'scientific_input_pins': scientific_pins, 'source_hashes': source_hashes, 'inputs': inputs, 'python': platform.python_version(), 'dependencies': dependencies}
    write_data_json(root / 'manifest.json', manifest)
    try:
        names = (('data',) if create_data else ()) + ('tables', 'figures', 'diagnostics')
        for name in names:
            (root / name).mkdir()
        write_data_json(root / 'config.json', config)
        yield root
        actual = artifact_files(root) - {'manifest.json'}
        if actual != expected_files:
            raise RuntimeError(f'artifact inventory mismatch; missing={sorted(expected_files - actual)}, unexpected={sorted(actual - expected_files)}')
        artifacts = []
        for relative in sorted(actual):
            path = root / relative
            if path.stat().st_size <= 0:
                raise RuntimeError(f'required artifact is empty: {relative}')
            artifacts.append({'path': relative, 'sha256': file_sha256(path), 'bytes': path.stat().st_size})
        manifest.update(status='complete', artifacts=artifacts, figure_exports=figure_exports_on_complete)
    except BaseException as exc:
        manifest.update(status='failed', error={'type': type(exc).__name__, 'message': str(exc)})
        raise
    finally:
        fd, name = tempfile.mkstemp(prefix='manifest-', suffix='.tmp', dir=root)
        try:
            with os.fdopen(fd, 'wb') as stream:
                stream.write(data_json_bytes(manifest))
            os.replace(name, root / 'manifest.json')
        finally:
            if os.path.exists(name):
                os.unlink(name)

def load_evaluation_summary(root: str | Path) -> dict:
    """Verify every artifact hash but load only the small report views."""
    root = Path(root).resolve(strict=True)
    analysis = root.parent.name
    if analysis not in {'contextual-anchor-q', 'contextual-dimsim'}:
        raise ValueError('unknown evaluation output analysis')
    name = 'anchor_q' if analysis == 'contextual-anchor-q' else 'dimsim'
    expected = {'config.json', f'data/{name}_batch.json', f'data/{name}_report.json', 'tables/tables.json', 'figures/plot_data.json', 'diagnostics/summary.json'}
    manifest = verify_artifact_inventory(root, analysis, root.name, expected)
    check_input_pins(manifest.get('inputs', []), manifest.get('scientific_input_pins', {}))
    report = read_finite_json(root / f'data/{name}_report.json')
    tables = read_finite_json(root / 'tables/tables.json')
    plot = read_finite_json(root / 'figures/plot_data.json')
    if tables != report.get('tables') or plot != report.get('plot_data'):
        raise ValueError('stored report views disagree')
    return {'root': root, 'manifest': manifest, 'report': report, 'tables': tables, 'plot_data': plot, 'config': read_finite_json(root / 'config.json')}

def load_evaluation_outputs(root: str | Path) -> dict:
    """Load a complete numerical run after checking exact inventory and hashes."""
    root = Path(root).resolve(strict=True)
    manifest_path = root / 'manifest.json'
    analysis = root.parent.name
    if analysis not in {'contextual-anchor-q', 'contextual-dimsim'}:
        raise ValueError('unknown evaluation output analysis')
    q = analysis.endswith('anchor-q')
    bn, rn = ('anchor_q_batch.json', 'anchor_q_report.json') if q else ('dimsim_batch.json', 'dimsim_report.json')
    expected = {'config.json', f'data/{bn}', f'data/{rn}', 'tables/tables.json', 'figures/plot_data.json', 'diagnostics/summary.json'}
    manifest = verify_artifact_inventory(root, analysis, root.name, expected)
    check_input_pins(manifest.get('inputs', []), manifest.get('scientific_input_pins', {}))
    if set(manifest['scientific_input_pins']) != {'bundle_manifest_sha256', 'model_ids', 'seeds', 'model_seed_provenance', 'analysis', 'run_id'}:
        raise ValueError('numerical output pins are malformed')
    batch, report = (read_finite_json(root / f'data/{bn}'), read_finite_json(root / f'data/{rn}'))
    tables, plot = (read_finite_json(root / 'tables/tables.json'), read_finite_json(root / 'figures/plot_data.json'))
    config = read_finite_json(root / 'config.json')
    read_finite_json(root / 'diagnostics/summary.json')
    if tables != report.get('tables') or plot != report.get('plot_data'):
        raise ValueError('stored tables or plot data differ from stored report')
    if batch.get('bundle_manifest_sha256') != manifest['scientific_input_pins']['bundle_manifest_sha256']:
        raise ValueError('stored batch bundle pin mismatch')
    results = batch.get('results')
    pins = manifest['scientific_input_pins']
    if not isinstance(results, list) or sorted({r.get('model_id') for r in results if isinstance(r, dict)}) != pins['model_ids']:
        raise ValueError('stored batch model inventory differs from manifest pins')
    if any((not isinstance(r, dict) or type(r.get('seed')) is not int or r.get('seed') not in pins['seeds'] for r in results)):
        raise ValueError('stored batch seed inventory is invalid')
    identity_pins = {(p.get('model_id'), p.get('seed')): p for p in pins['model_seed_provenance'] if isinstance(p, dict)}
    expected_pairs = {(model, seed) for model in pins['model_ids'] for seed in pins['seeds']}
    if len(identity_pins) != len(pins['model_seed_provenance']) or set(identity_pins) != expected_pairs:
        raise ValueError('manifest model/seed provenance inventory is invalid')
    for row in results:
        pin = identity_pins[row['model_id'], row['seed']]
        if any((row.get(k) != pin.get(k) for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256'))):
            raise ValueError('stored batch provenance differs from manifest pins')
        if row.get('experiment_hashes') is not None and row['experiment_hashes'] != pin.get('experiment_hashes'):
            raise ValueError('stored batch experiment hashes differ from manifest pins')
    if config.get('reference_model') is not None and report.get('reference_model') is not None and (report.get('reference_model') != config['reference_model']):
        raise ValueError('stored report reference metadata differs from config')
    return {'root': root, 'manifest': manifest, 'config': config, 'batch': batch, 'report': report, 'tables': tables, 'plot_data': plot}

def load_random_control_summary(root: str | Path) -> dict:
    """Read hash-verified random-control summaries without loading vector matrices."""
    root = Path(root).resolve(strict=True)
    expected = {'config.json', 'data/occurrence_ids.json', 'data/control_report.json', 'tables/tables.json', 'figures/plot_data.json', 'diagnostics/summary.json', *(f'data/random_seed_{seed}.npy' for seed in range(5))}
    manifest = verify_artifact_inventory(root, 'contextual-random-control', root.name, expected)
    pins = manifest.get('scientific_input_pins', {})
    check_input_pins(manifest.get('inputs', []), pins)
    if set(pins) != {'bundle_manifest_sha256', 'seed_kind', 'random_seeds', 'dimension'} or pins['seed_kind'] != 'random' or pins['random_seeds'] != list(range(5)) or (type(pins['dimension']) is not int) or (pins['dimension'] <= 0):
        raise ValueError('random-control scientific pins malformed')
    report = read_finite_json(root / 'data/control_report.json')
    if report.get('schema_version') != 1 or report.get('model_id') != 'random-control' or report.get('seed_kind') != 'random' or (report.get('random_seeds') != list(range(5))) or (report.get('bundle_manifest_sha256') != pins['bundle_manifest_sha256']):
        raise ValueError('stored random-control identity mismatch')
    tables = read_finite_json(root / 'tables/tables.json')
    plot = read_finite_json(root / 'figures/plot_data.json')
    if tables != {'anchor_q': report.get('anchor_q'), 'dimsim': report.get('dimsim')} or plot != tables:
        raise ValueError('random-control table or plot payload mismatch')
    return {'root': root, 'manifest': manifest, 'config': read_finite_json(root / 'config.json'), 'report': {key: value for key, value in report.items() if key != 'per_seed'}, 'tables': tables, 'plot_data': plot}


### Immutable residual outputs


In [ ]:
def write_interpretation_outputs(project_root, run_id, result, *, input_roots, source_hashes):
    """Persist a pure analysis result with explicit input identities and code pins.

    Callers must hash-verify source artifacts before building the result. Input
    manifests and supplied source hashes are preserved by the shared lifecycle.
    """
    tables = readable_tables(result)
    expected = {'config.json', 'data/interpretation.json', 'diagnostics/summary.json'}
    expected.update(('tables/' + name for name in tables))
    expected.update(('data/top-50-' + model + '.json' for model in result['top_pairs']))
    with presentation_run(project_root, 'contextual-interpretation', run_id, input_roots=input_roots, config=result['config'], source_hashes=source_hashes, expected_files=expected, scientific_pins={'bundle_manifest_sha256': result['bundle_manifest_sha256'], 'model_seed_provenance': result['provenance']}) as root:
        write_data_json(root / 'data/interpretation.json', result)
        write_data_json(root / 'diagnostics/summary.json', result['diagnostics'])
        for model, table in result['top_pairs'].items():
            write_data_json(root / f'data/top-50-{model}.json', table)
        for name, text in tables.items():
            with (root / 'tables' / name).open('x', encoding='utf-8', newline='\n') as stream:
                stream.write(text)
    verify_artifact_inventory(Path(root), 'contextual-interpretation', run_id, expected)
    return root


### Verified saved-score loading


In [ ]:
def _decode(text):

    def pairs(items):
        result = {}
        for key, value in items:
            if key in result:
                raise ValueError('duplicate JSON key')
            result[key] = value
        return result

    def invalid(value):
        raise ValueError(f'non-finite JSON constant: {value}')
    return json.loads(text, object_pairs_hook=pairs, parse_constant=invalid)

class _Reader:
    """Read JSON values using structural scanning, independent of whitespace.

    Only one result object is decoded at a time. The enclosing results array
    is never materialized. Natural scores within that object are then released.
    """
    tokens = re.compile('[{}\\[\\]"\\\\]')

    def __init__(self, stream, chunk_size=1024 * 1024):
        self.stream, self.chunk_size = (stream, chunk_size)
        self.buffer, self.pos = ('', 0)

    def peek(self):
        while True:
            if self.pos == len(self.buffer):
                self.buffer, self.pos = (self.stream.read(self.chunk_size), 0)
                if not self.buffer:
                    return ''
            if self.buffer[self.pos].isspace():
                self.pos += 1
            else:
                return self.buffer[self.pos]

    def expect(self, token):
        if self.peek() != token:
            raise ValueError(f'expected JSON delimiter {token}')
        self.pos += 1

    def value(self):
        first = self.peek()
        if not first:
            raise ValueError('truncated JSON value')
        if first not in '{["':
            parts = []
            while True:
                start = self.pos
                while self.pos < len(self.buffer) and self.buffer[self.pos] not in ',]} \t\r\n':
                    self.pos += 1
                parts.append(self.buffer[start:self.pos])
                if self.pos < len(self.buffer):
                    break
                self.buffer, self.pos = (self.stream.read(self.chunk_size), 0)
                if not self.buffer:
                    break
            return _decode(''.join(parts))
        depth, quoted, escaped, parts = (0, False, False, [])
        start = self.pos
        while True:
            for match in self.tokens.finditer(self.buffer, self.pos):
                index, token = (match.start(), match.group())
                if quoted:
                    if escaped and index == self.pos:
                        escaped = False
                        self.pos = index + 1
                        continue
                    escaped = False
                    if token == '\\':
                        escaped = True
                    elif token == '"':
                        quoted = False
                elif token == '"':
                    quoted = True
                elif token in '{[':
                    depth += 1
                elif token in '}]':
                    depth -= 1
                self.pos = index + 1
                if not quoted and depth == 0:
                    parts.append(self.buffer[start:self.pos])
                    return _decode(''.join(parts))
            escaped = escaped and self.pos == len(self.buffer)
            parts.append(self.buffer[start:])
            self.buffer, self.pos, start = (self.stream.read(self.chunk_size), 0, 0)
            if not self.buffer:
                raise ValueError('truncated JSON container')

def iter_dimsim_records(path, *, bundle_sha256, chunk_size=1024 * 1024):
    """Strictly stream the top-level batch object, including its trailing syntax."""
    with Path(path).open(encoding='utf-8') as stream:
        reader = _Reader(stream, chunk_size)
        reader.expect('{')
        keys, header = (set(), {})
        while reader.peek() != '}':
            key = reader.value()
            if not isinstance(key, str) or key in keys:
                raise ValueError('invalid or duplicate batch key')
            keys.add(key)
            reader.expect(':')
            if key == 'results':
                reader.expect('[')
                while reader.peek() != ']':
                    record = reader.value()
                    if not isinstance(record, dict):
                        raise ValueError('result must be an object')
                    yield record
                    del record
                    if reader.peek() == ']':
                        break
                    reader.expect(',')
                    if reader.peek() == ']':
                        raise ValueError('trailing results comma')
                reader.expect(']')
            else:
                header[key] = reader.value()
            if reader.peek() == '}':
                break
            reader.expect(',')
            if reader.peek() == '}':
                raise ValueError('trailing batch comma')
        reader.expect('}')
        if reader.peek():
            raise ValueError('trailing content after batch')
        if keys != {'results', 'schema_version', 'bundle_manifest_sha256'} or type(header.get('schema_version')) is not int or header['schema_version'] != 1 or (header.get('bundle_manifest_sha256') != bundle_sha256):
            raise ValueError('batch header identity mismatch')

def _artifact_path(root, name):
    path = (root / name).resolve()
    if not isinstance(name, str) or path == root or root not in path.parents or (not path.is_file()):
        raise ValueError('unsafe or missing bundle artifact')
    return path

def load_saved_q_batch(root):
    """Read Q scores once, retaining only fields used by descriptive summaries."""
    summary = load_evaluation_summary(root)
    manifest = summary['manifest']
    if manifest['analysis'] != 'contextual-anchor-q':
        raise ValueError('Anchor-Q input required')
    sha = manifest['scientific_input_pins']['bundle_manifest_sha256']
    pins = {(p['model_id'], p['seed']): p for p in manifest['scientific_input_pins']['model_seed_provenance']}
    records = []
    for record in iter_dimsim_records(Path(root) / 'data/anchor_q_batch.json', bundle_sha256=sha):
        pin = pins.get((record.get('model_id'), record.get('seed')))
        if pin is None or any((record.get(k) != pin.get(k) for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256'))):
            raise ValueError('Q record provenance differs from manifest')
        record['score']['draw_scores'] = [{k: r[k] for k in ('draw_id', 'triplet_id', 'edge_id', 'score')} for r in record['score']['draw_scores']]
        records.append(record)
    return {'schema_version': 1, 'bundle_manifest_sha256': sha, 'results': records}

def run_saved_interpretation(*, dimsim_root, dimsim_manifest_sha256, bundle_path, bundle_manifest_sha256, project_root, run_id, source_hashes):
    """Verify exact inputs, stream saved scores, analyze, and write a new run.

    No calls to model, extraction, sampling or scoring APIs occur. All parsing
    and identity validation finishes before the output directory is created.
    """
    dimsim_root, bundle_path = (Path(p).resolve(strict=True) for p in (dimsim_root, bundle_path))
    target = Path(project_root).resolve() / 'outputs/contextual-interpretation' / safe_run_id(run_id)
    ensure_output_outside_inputs(target, [dimsim_root, bundle_path])
    if target.exists():
        raise FileExistsError(f'refusing to overwrite existing output: {target}')
    for parent in (target, *target.parents):
        if (parent / 'manifest.json').exists():
            raise ValueError('output would modify an immutable run')
    for sha, path in ((dimsim_manifest_sha256, dimsim_root / 'manifest.json'), (bundle_manifest_sha256, bundle_path / 'manifest.json')):
        if not is_sha256(sha) or file_sha256(path) != sha:
            raise ValueError('explicit input manifest hash mismatch')
    expected = {'config.json', 'data/dimsim_batch.json', 'data/dimsim_report.json', 'diagnostics/summary.json', 'figures/plot_data.json', 'tables/tables.json'}
    manifest = verify_artifact_inventory(dimsim_root, 'contextual-dimsim', dimsim_root.name, expected)
    pins = manifest['scientific_input_pins']
    if pins.get('bundle_manifest_sha256') != bundle_manifest_sha256 or set(pins.get('model_ids', [])) != set(MODELS) or pins.get('seeds') != list(SEEDS):
        raise ValueError('DIMSIM scientific pins mismatch')
    pin_rows = pins.get('model_seed_provenance', [])
    pin_map = {(r['model_id'], r['seed']): r for r in pin_rows}
    if len(pin_rows) != 25 or set(pin_map) != {(m, s) for m in MODELS for s in SEEDS}:
        raise ValueError('invalid model/seed provenance inventory')
    bundle_manifest = read_finite_json(bundle_path / 'manifest.json')
    if bundle_manifest.get('schema_version') != 3:
        raise ValueError('schema-3 bundle required')
    artifacts = bundle_manifest['artifacts']
    for name, sha in artifacts.items():
        if not is_sha256(sha) or file_sha256(_artifact_path(bundle_path, name)) != sha:
            raise ValueError(f'bundle artifact hash mismatch: {name}')

    def jsonl(name):
        if name not in artifacts:
            raise ValueError(f'bundle inventory missing {name}')
        with _artifact_path(bundle_path, name).open(encoding='utf-8') as stream:
            return [_decode(line) for line in stream]
    occurrences, sentences = (jsonl('occurrences.jsonl'), jsonl('sentences.jsonl'))
    pairs = jsonl('pairs/dimsim-balanced.jsonl')
    identity_keys = ('draw_id', 'pair_id', 'left_id', 'right_id', 'overlap_mask', 'dimsim_distance')
    identities = [tuple((p[k] for k in identity_keys)) for p in pairs]

    def records():
        for record in iter_dimsim_records(dimsim_root / 'data/dimsim_batch.json', bundle_sha256=bundle_manifest_sha256):
            pin = pin_map.get((record.get('model_id'), record.get('seed')))
            if pin is None or any((record.get(k) != pin.get(k) for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256'))):
                raise ValueError('record provenance differs from manifest')
            rows = record['score']['modes']['dimsim-balanced']['rows']
            if [tuple((r[k] for k in identity_keys)) for r in rows] != identities:
                raise ValueError('balanced scores differ from fixed bundle sample')
            compact = {k: record[k] for k in ('model_id', 'seed', 'run_id', 'run_manifest_sha256', 'checkpoint_sha256')}
            score = record['score']
            compact['score'] = {k: score[k] for k in ('model_id', 'seed', 'bundle_manifest_sha256')}
            compact['score']['modes'] = {'dimsim-balanced': {'rows': rows}}
            del record, score
            yield compact
    result = build_interpretation(records(), occurrences, sentences, bundle_manifest_sha256=bundle_manifest_sha256)
    result['config']['saved_inputs'] = {'dimsim_root': str(dimsim_root), 'dimsim_manifest_sha256': dimsim_manifest_sha256, 'bundle_path': str(bundle_path), 'bundle_manifest_sha256': bundle_manifest_sha256, 'run_id': run_id, 'loader': 'one JSON result at a time; balanced projection'}
    batch_artifact = next((a for a in manifest['artifacts'] if a['path'] == 'data/dimsim_batch.json'))
    if file_sha256(dimsim_root / batch_artifact['path']) != batch_artifact['sha256']:
        raise ValueError('score bytes changed during analysis')
    for name in ('occurrences.jsonl', 'sentences.jsonl', 'pairs/dimsim-balanced.jsonl'):
        if file_sha256(_artifact_path(bundle_path, name)) != artifacts[name]:
            raise ValueError('bundle bytes changed during analysis')
    if file_sha256(dimsim_root / 'manifest.json') != dimsim_manifest_sha256 or file_sha256(bundle_path / 'manifest.json') != bundle_manifest_sha256:
        raise ValueError('input identity changed during analysis')
    return write_interpretation_outputs(project_root, run_id, result, input_roots=[bundle_path, dimsim_root], source_hashes=source_hashes)


### Q12/9/3 and residual quantiles


In [ ]:
def build_q_group_tables(batch):
    """Equal transition weights; MC precision retains matching seed draws."""
    report = build_anchor_q_report(batch, reference_model='syllable')
    if {r['model_id'] for r in batch['results']} != set(MODELS):
        raise ValueError('Q groups require the five learned models')
    arrays = {}
    for row in batch['results']:
        _, edges = _edge_ids(row['score']['draw_scores'], list(EDGE_ORDER))
        arrays[row['model_id'], row['seed']] = {e: np.asarray(v, dtype=float) for e, v in edges.items()}
    groups = {'all12': list(EDGE_ORDER), 'partial9': [e for e in EDGE_ORDER if not e.endswith('->111')], 'full3': [e for e in EDGE_ORDER if e.endswith('->111')]}
    seed_rows, summary, deltas, seed_deltas = ([], [], [], [])

    def estimate(values):
        means = [float(a.mean()) for a in values]
        se = math.sqrt(sum((float(a.var(ddof=1)) / a.size for a in values))) / len(values) if all((a.size > 1 for a in values)) else None
        return (float(np.mean(means)), se)
    for group, edges in groups.items():
        for metric in ('initial', 'final', 'tone', 'overall'):
            selected = [e for e in edges if metric == 'overall' or e.startswith(metric + ':')]
            for model in MODELS:
                qs, diffs = ([], [])
                for seed in SEEDS:
                    q, se = estimate([arrays[model, seed][e] for e in selected])
                    delta, dse = estimate([arrays[model, seed][e] - arrays['syllable', seed][e] for e in selected])
                    qs.append(q)
                    diffs.append(delta)
                    seed_rows.append({'group': group, 'metric': metric, 'model_id': model, 'seed': seed, 'q': q, 'mc_se': se})
                    seed_deltas.append({'group': group, 'metric': metric, 'model_id': model, 'seed': seed, 'delta_q': delta, 'mc_se': dse})
                _, se = estimate([np.mean([arrays[model, s][e] for s in SEEDS], axis=0) for e in selected])
                _, dse = estimate([np.mean([arrays[model, s][e] - arrays['syllable', s][e] for s in SEEDS], axis=0) for e in selected])
                base = {'group': group, 'metric': metric, 'model_id': model, 'transition_count': len(selected), 'transition_weight': 1 / len(selected)}
                summary.append({**base, 'mean': float(np.mean(qs)), 'sample_sd': float(np.std(qs, ddof=1)), 'mc_se': se, 'seed_values': qs})
                deltas.append({**base, 'reference_model': 'syllable', 'mean': float(np.mean(diffs)), 'sample_sd': float(np.std(diffs, ddof=1)), 'mc_se': dse, 'seed_values': diffs})
    dispersions = {(r['model_id'], r['edge_id']): r['sample_sd'] for r in report['tables']['model_edge_seed_summary']}
    forest = [{**r, 'sample_sd': dispersions[r['model_id'], r['edge_id']]} for r in report['plot_data']['edges']]
    return {'config': {'groups': groups, 'weights': 'equal transitions within each group and metric', 'seed_ids': list(SEEDS), 'reference_model': 'syllable', 'mc_se': 'matched draw seed averages; stratified independent transitions', 'bootstrap': False, 'corpus_ci': False}, 'seed_metrics': seed_rows, 'summary': summary, 'paired_deltas': deltas, 'seed_paired_deltas': seed_deltas, 'forest': forest}

def build_residual_qq_data(residuals, *, max_quantiles=401):
    """Per-seed sample-SD standardization; retain both extreme order statistics."""
    if type(max_quantiles) is not int or max_quantiles < 2:
        raise ValueError('max_quantiles must be an integer >= 2')
    if set(residuals) != {*MODELS, 'dimsim'}:
        raise ValueError('six residual outcomes are required')
    curves = []
    normal = NormalDist()
    size = None
    for model in (*MODELS, 'dimsim'):
        seeds = {None: residuals[model]} if model == 'dimsim' else residuals[model]
        if model != 'dimsim' and set(seeds) != {str(s) for s in SEEDS}:
            raise ValueError('exact five separate residual seeds are required')
        for seed, values in seeds.items():
            y = np.asarray(values, dtype=float)
            if y.ndim != 1 or not len(y) or (not np.isfinite(y).all()):
                raise ValueError('finite nonempty residual vectors are required')
            if size is None:
                size = len(y)
            if len(y) != size:
                raise ValueError('residual populations differ')
            sd = float(np.std(y, ddof=1)) if len(y) > 1 else 0.0
            reason = 'fewer_than_two_draws' if len(y) < 2 else 'constant_residual' if sd == 0 else None
            row = {'model_id': model, 'seed': None if seed is None else int(seed), 'draw_count': len(y), 'residual_mean': float(y.mean()), 'sample_sd': sd, 'status': reason or 'ok', 'normal_quantiles': [], 'standardized_residuals': []}
            if reason is None:
                idx = np.unique(np.rint(np.linspace(0, len(y) - 1, min(len(y), max_quantiles))).astype(int))
                row['normal_quantiles'] = [normal.inv_cdf((int(i) + 0.5) / len(y)) for i in idx]
                row['standardized_residuals'] = ((np.sort(y)[idx] - y.mean()) / sd).tolist()
            curves.append(row)
    return {'config': {'max_quantiles': max_quantiles, 'plotting_positions': '(rank + 0.5) / n', 'standardization': 'within seed; sample SD', 'extremes_retained': True, 'sample': 'dimsim-balanced', 'draw_multiplicity': 'preserved'}, 'curves': curves}

def build_interpretation_presentation_data(q_batch, interpretation):
    """Prepare tables and plot data once; the renderer never refits outcomes."""
    if q_batch['bundle_manifest_sha256'] != interpretation['bundle_manifest_sha256']:
        raise ValueError('Q and residual bundle pins differ')
    q_pins = {(r['model_id'], r['seed']): {k: r[k] for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256')} for r in q_batch['results']}
    expected = {(m, p['seed']): {k: p[k] for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256')} for m, rows in interpretation['provenance'].items() for p in rows}
    if q_pins != expected or len(q_batch['results']) != 25:
        raise ValueError('Q and residual training provenance differ')
    q = build_q_group_tables(q_batch)
    tables = readable_tables(interpretation)
    for name, rows in [('anchor-q-groups', q['summary']), ('anchor-q-group-differences', q['paired_deltas'])]:
        lines = ['| Model | Group | Metric | Mean | Seed SD | MC SE |', '| --- | --- | --- | --- | --- | --- |']
        for row in rows:
            se = '—' if row['mc_se'] is None else f"{row['mc_se']:.6f}"
            lines.append(f"| {row['model_id']} | {row['group']} | {row['metric']} | {row['mean']:.6f} | {row['sample_sd']:.6f} | {se} |")
        tables[name + '.md'] = '\n'.join(lines) + '\n'
    return {'schema_version': 1, 'bundle_manifest_sha256': interpretation['bundle_manifest_sha256'], 'provenance': interpretation['provenance'], 'q_groups': q, 'regression_summary': interpretation['summary'], 'regression_per_seed': interpretation['per_seed'], 'top_pairs': interpretation['top_pairs'], 'residual_qq': build_residual_qq_data(interpretation['residuals']), 'tables': tables}

def write_interpretation_presentation_data(project_root, run_id, prepared, *, input_roots, source_hashes):
    """Persist derived data separately from numerical and presentation runs."""
    expected = {'config.json', 'data/presentation_data.json', 'data/top_pairs.json', 'diagnostics/summary.json'}
    expected.update(('tables/' + name for name in prepared['tables']))
    with presentation_run(project_root, 'contextual-interpretation-data', run_id, input_roots=input_roots, source_hashes=source_hashes, config={'q': prepared['q_groups']['config'], 'qq': prepared['residual_qq']['config']}, expected_files=expected, scientific_pins={'evaluation_bundle_manifest_sha256': prepared['bundle_manifest_sha256'], 'model_seed_provenance': prepared['provenance']}) as root:
        write_data_json(root / 'data/presentation_data.json', prepared)
        write_data_json(root / 'data/top_pairs.json', prepared['top_pairs'])
        write_data_json(root / 'diagnostics/summary.json', {'table_count': len(prepared['tables']), 'residual_curve_count': len(prepared['residual_qq']['curves'])})
        for name, body in prepared['tables'].items():
            if '/' in name or '\\' in name or (not name.endswith('.md')):
                raise ValueError('unsafe table filename')
            with (root / 'tables' / name).open('x', encoding='utf-8', newline='\n') as stream:
                stream.write(body)
    return root


### Readable table export


In [ ]:
def guard_table_output(target):
    path = Path(target).resolve()
    for ancestor in (path, *path.parents):
        manifest = ancestor / 'manifest.json'
        if not manifest.is_file():
            continue
        try:
            data = json.loads(manifest.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if data.get('scope') == 'numerical_outputs' and data.get('analysis') in {'contextual-anchor-q', 'contextual-dimsim'} and (data.get('status') in {'complete', 'failed'}):
            raise ValueError(f'Refusing to write inside immutable evaluation run: {ancestor}')

def format_table_number(v):
    return '—' if v is None else str(v) if isinstance(v, int) and (not isinstance(v, bool)) else f'{v:.3f}'

def markdown_table(title, headers, rows):
    out = [f'# {title}', '', '| ' + ' | '.join(headers) + ' |', '|' + '|'.join(('---' for _ in headers)) + '|']
    out += ['| ' + ' | '.join((str(x) for x in row)) + ' |' for row in rows]
    return '\n'.join(out) + '\n'

def render_evaluation_tables(anchor_tables: dict, dimsim_tables: dict, output_dir):
    """Write Markdown tables from report tables without recalculating summaries."""
    out = Path(output_dir)
    guard_table_output(out)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError('Table output directory must be empty')
    out.mkdir(parents=True, exist_ok=True)
    artifacts = {}

    def write(name, body):
        if Path(name).name != name or not name.endswith('.md'):
            raise ValueError('Unsafe table filename')
        (out / name).write_text(body, encoding='utf-8')
        artifacts[name] = body
    models = sorted({r['model_id'] for r in anchor_tables['model_seed_edges']})
    for model in models:
        for seed in range(5):
            rows = [r for r in anchor_tables['model_seed_edges'] if r['model_id'] == model and r['seed'] == seed]
            fields = ('edge_id', 'q', 'mc_se', 'draw_count', 'unique_triplets', 'unique_anchor_occurrences', 'unique_anchor_syllables', 'unique_occurrences')
            write(f'anchor_q_edges_{model}_seed_{seed}.md', markdown_table(f'Anchor-Q edges: {model}, seed {seed}', list(fields), [[r['edge_id'], format_table_number(r['q']), format_table_number(r['mc_se']), r['draw_count'], *[(r.get('coverage') or {}).get(k, '—') for k in fields[4:]]] for r in rows]))
        metric_rows = [r for r in anchor_tables['model_seed_metrics'] if r['model_id'] == model]
        summary = anchor_tables.get('model_metric_seed_summary', [])
        fields = ('metric', 'seed', 'q', 'mc_se')
        write(f'anchor_q_metrics_{model}.md', markdown_table(f'Anchor-Q metrics: {model}', list(fields), [[r['metric'], r['seed'], format_table_number(r['q']), format_table_number(r['mc_se'])] for r in metric_rows]))
        sm = [r for r in summary if r['model_id'] == model]
        write(f'anchor_q_metric_summary_{model}.md', markdown_table(f'Anchor-Q mean and sample SD: {model}', ['Metric', 'Mean', 'Sample SD', 'MC SE'], [[r['metric'], format_table_number(r['mean']), format_table_number(r['sample_sd']), format_table_number(r['mc_se'])] for r in sm]))
        erows = [r for r in anchor_tables.get('model_edge_seed_summary', []) if r['model_id'] == model]
        write(f'anchor_q_edge_summary_{model}.md', markdown_table(f'Anchor-Q edge summary: {model}', ['Edge', 'Mean', 'Sample SD', 'MC SE'], [[r['edge_id'], format_table_number(r['mean']), format_table_number(r['sample_sd']), format_table_number(r['mc_se'])] for r in erows]))
        metric_names = ('initial', 'final', 'tone', 'overall')
        by_seed = {seed: {r['metric']: format_table_number(r['q']) for r in anchor_tables['model_seed_metrics'] if r['model_id'] == model and r['seed'] == seed} for seed in range(5)}
        smap = {r['metric']: r for r in anchor_tables.get('model_metric_seed_summary', []) if r['model_id'] == model}
        rows = [[f'Seed {seed}'] + [by_seed[seed].get(metric, '—') for metric in metric_names] for seed in range(5)]
        rows.append(['Mean ± sample SD'] + [f"{format_table_number(smap[m]['mean'])} ± {format_table_number(smap[m]['sample_sd'])}" for m in metric_names if m in smap])
        write(f'anchor_q_metrics_wide_{model}.md', markdown_table(f'Anchor-Q metrics by seed: {model}', ['Seed', *metric_names], rows))
    for key, title, fields in (('model_seed_edges', 'Anchor-Q edge values', ('model_id', 'seed', 'edge_id', 'q', 'mc_se', 'draw_count')), ('model_seed_metrics', 'Anchor-Q metric values', ('model_id', 'seed', 'metric', 'q', 'mc_se')), ('global_metric_summary', 'Anchor-Q global summaries', ('model_id', 'metric', 'mean', 'sample_sd', 'mc_se')), ('paired_deltas', 'Paired Anchor-Q differences', ('model_id', 'reference_model', 'metric', 'mean_delta_q', 'sample_sd_seed_delta', 'mc_se')), ('transition_shares', 'Anchor-Q transition shares', ('model_id', 'edge_id', 'share', 'status')), ('component_shares', 'Anchor-Q component shares', ('model_id', 'component', 'share', 'status'))):
        rows = [[r.get(f) if isinstance(r.get(f), str) else format_table_number(r.get(f)) for f in fields] for r in anchor_tables[key]]
        write(f'{key}.md', markdown_table(title, list(fields), rows))
    paired = []
    for r in anchor_tables['paired_deltas']:
        paired.append([r['model_id'], r['metric'], f"{format_table_number(r['mean_delta_q'])} ({format_table_number(r['sample_sd_seed_delta'])}; {format_table_number(r['mc_se'])})"])
    write('anchor_q_paired_delta_cells.md', markdown_table('Paired Anchor-Q differences', ['Model', 'Metric', 'Mean ΔQ (sample SD; MC SE)'], paired))
    metric_names = ('initial', 'final', 'tone', 'overall')
    paired_map = {(r['model_id'], r['metric']): r for r in anchor_tables['paired_deltas']}
    paired_models = sorted({r['model_id'] for r in anchor_tables['paired_deltas']})
    paired_wide = [[model] + [f"{format_table_number(paired_map[model, m]['mean_delta_q'])} ({format_table_number(paired_map[model, m]['sample_sd_seed_delta'])}; {format_table_number(paired_map[model, m]['mc_se'])})" for m in metric_names] for model in paired_models]
    write('anchor_q_paired_deltas_wide.md', markdown_table('Paired Anchor-Q differences', ['Model', *metric_names], paired_wide))
    global_map = {(r['model_id'], r['metric']): r for r in anchor_tables['global_metric_summary']}
    global_models = sorted({r['model_id'] for r in anchor_tables['global_metric_summary']})
    global_wide = [[model] + [f"{format_table_number(global_map[model, m]['mean'])} ± {format_table_number(global_map[model, m]['sample_sd'])}" for m in metric_names] + [format_table_number(global_map[model, m]['mc_se']) for m in metric_names] for model in global_models]
    write('anchor_q_global_summary_wide.md', markdown_table('Anchor-Q global summary', ['Model', *[f'{m} mean ± SD' for m in metric_names], *[f'{m} MC SE' for m in metric_names]], global_wide))
    for key, title, fields in (('seed_rho', 'DIMSIM full-range rho by seed', ('model_id', 'seed', 'mode', 'rho', 'valid_draw_count', 'unique_pair_count', 'status')), ('model_rho', 'DIMSIM full-range rho summaries', ('model_id', 'mode', 'mean', 'sample_sd', 'status')), ('paired_rho_deltas', 'Paired DIMSIM rho differences', ('model_id', 'reference_model', 'mode', 'mean', 'sample_sd', 'status')), ('seed_boundary', 'Exploratory boundary thresholds', ('model_id', 'seed', 'mode', 'threshold', 'status')), ('bins', 'DIMSIM curve bins and support', ('model_id', 'mode', 'bin', 'mean_cosine', 'supported'))):
        rows = []
        for r in dimsim_tables[key]:
            row = []
            for f in fields:
                v = r.get(f)
                if f == 'bin' and isinstance(r.get('seed_points'), list):
                    pts = r['seed_points']
                    v = f"{v}; draws={pts[0].get('draw_count')}; unique={pts[0].get('unique_pair_count')}" if pts else v
                row.append(v if isinstance(v, (str, bool)) else format_table_number(v))
            rows.append(row)
        write(f'{key}.md', markdown_table(title, list(fields), rows))
    if 'model_boundary' in dimsim_tables:
        rows = [[r.get('model_id'), r.get('mode'), format_table_number(r.get('mean')), format_table_number(r.get('sample_sd')), format_table_number(r.get('median')), r.get('status')] for r in dimsim_tables['model_boundary']]
        write('dimsim_boundary_summary.md', markdown_table('Exploratory DIMSIM boundary summary', ['Model', 'Mode', 'Mean', 'Sample SD', 'Median', 'Status'], rows))
    if 'model_rho' in dimsim_tables:
        modes = sorted({r['mode'] for r in dimsim_tables['model_rho']})
        rho_map = {(r['model_id'], r['mode']): r for r in dimsim_tables['model_rho']}
        dmodels = sorted({r['model_id'] for r in dimsim_tables['model_rho']})
        wide = [[model] + [f"{format_table_number(rho_map[model, mode]['mean'])} ± {format_table_number(rho_map[model, mode]['sample_sd'])}" for mode in modes] for model in dmodels]
        write('dimsim_model_rho_wide.md', markdown_table('DIMSIM full-range rho by model and mode', ['Model', *modes], wide))
    return sorted(out.glob('*.md'))


### Shared figure protocol


In [ ]:
ROOT = STYLE_RESOURCE_ROOT
STYLE_PATH = ROOT / 'style.toml'
REGISTRY_PATH = ROOT / 'registry.toml'
MM_PER_INCH = 25.4

class VisualizationError(ValueError):
    """An actionable protocol violation."""
STYLE_KEYS = {'schema': {'version'}, 'font': {'family', 'axis_label_pt', 'tick_pt', 'legend_pt', 'panel_heading_pt', 'panel_letter_pt', 'annotation_pt', 'normal_weight', 'panel_letter_weight'}, 'color': {'background', 'foreground', 'secondary', 'tertiary', 'quaternary'}, 'line': {'data_pt', 'axes_pt', 'reference_pt', 'interval_pt', 'caps_pt'}, 'marker': {'size_pt', 'edge_pt', 'face', 'markevery_target'}, 'layout': {'single_width_mm', 'multi_width_mm', 'panel_aspect', 'left_mm', 'right_mm', 'bottom_mm', 'top_mm', 'hgap_mm', 'vgap_mm', 'legend_row_mm', 'legend_gap_mm', 'default_columns'}, 'axes': {'grid', 'minor_ticks', 'tick_direction', 'major_length_pt', 'major_width_pt', 'major_pad_pt', 'major_target', 'panel_label_x_mm', 'panel_label_y_mm'}, 'legend': {'max_per_row', 'handle_length_em', 'column_spacing_em', 'location'}, 'histogram': {'density', 'fill', 'bootstrap_bins'}, 'export': {'formats', 'dpi', 'transparent', 'overwrite'}, 'backend': {'rcparams'}}
FIGURE_KEYS = {'status', 'kind', 'name', 'xlabel', 'ylabel', 'series', 'order', 'scale', 'bin_rule', 'bins', 'density', 'columns', 'xlim', 'ylim', 'edges_start', 'edges_stop', 'edges_count'}
SEMANTIC_KEYS = {'label', 'color', 'linestyle', 'marker'}
ALLOWED_RCPARAMS = {'pdf.fonttype', 'ps.fonttype', 'svg.fonttype'}

def read_style_toml(p):
    raw = Path(p).read_bytes()
    return (tomllib.loads(raw.decode()), raw)

def reject_unknown_style_keys(d, a, w):
    e = set(d) - a
    if e:
        raise VisualizationError(f"Unknown key(s) in {w}: {', '.join(sorted(e))}")

def require_positive_style_value(v, n):
    if isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (v <= 0):
        raise VisualizationError(f'{n} must be finite and positive')

@dataclass(frozen=True)
class Protocol:
    style: dict[str, Any]
    registry: dict[str, Any]
    style_hash: str
    registry_hash: str
    style_path: Path
    registry_path: Path

    def figure(self, fid):
        try:
            return self.registry['figures'][fid]
        except KeyError as e:
            raise VisualizationError(f'Unknown figure ID: {fid}') from e

def load_protocol(style_path=STYLE_PATH, registry_path=REGISTRY_PATH):
    sp = Path(style_path).resolve()
    rp = Path(registry_path).resolve()
    s, sr = read_style_toml(sp)
    r, rr = read_style_toml(rp)
    reject_unknown_style_keys(s, set(STYLE_KEYS), 'style root')
    for sec, keys in STYLE_KEYS.items():
        if sec not in s:
            raise VisualizationError(f'Missing style section: {sec}')
        reject_unknown_style_keys(s[sec], keys, f'style.{sec}')
        missing = keys - set(s[sec])
        if missing:
            raise VisualizationError(f"Missing key(s) in style.{sec}: {', '.join(sorted(missing))}")
    if s['schema']['version'] != 1:
        raise VisualizationError('Unsupported style schema version')
    if s['font']['family'] != 'Arial':
        raise VisualizationError('The protocol requires Arial')
    for k, v in s['font'].items():
        if k.endswith('_pt'):
            require_positive_style_value(v, f'font.{k}')
    if (s['font']['normal_weight'], s['font']['panel_letter_weight']) != ('normal', 'bold'):
        raise VisualizationError('Font weights must be normal and bold')
    for k, v in s['layout'].items():
        if k != 'default_columns':
            require_positive_style_value(v, f'layout.{k}')
    if s['layout']['default_columns'] != [1, 2, 2, 2, 3, 3, 3]:
        raise VisualizationError('Invalid default columns')
    a = s['axes']
    if a['grid'] is not False or a['minor_ticks'] is not False or a['tick_direction'] != 'in':
        raise VisualizationError('Axes require grid/minor false and inward ticks')
    for k in ('major_length_pt', 'major_width_pt', 'major_pad_pt', 'major_target'):
        require_positive_style_value(a[k], f'axes.{k}')
    if s['export']['formats'] != ['png', 'eps'] or s['export']['dpi'] != 300 or s['export']['transparent'] is not False:
        raise VisualizationError('Export requires opaque PNG300 then EPS')
    reject_unknown_style_keys(s['backend']['rcparams'], ALLOWED_RCPARAMS, 'style.backend.rcparams')
    reject_unknown_style_keys(r, {'schema', 'figures', 'semantics'}, 'registry root')
    if r.get('schema', {}).get('version') != 1 or not isinstance(r.get('figures'), dict) or (not isinstance(r.get('semantics'), dict)):
        raise VisualizationError('Invalid registry schema')
    for sid, sem in r['semantics'].items():
        reject_unknown_style_keys(sem, SEMANTIC_KEYS, f'semantics.{sid}')
        if set(sem) != SEMANTIC_KEYS or not all((isinstance(x, str) and x.strip() for x in sem.values())):
            raise VisualizationError(f'Invalid semantics.{sid}')
    for fid, f in r['figures'].items():
        reject_unknown_style_keys(f, FIGURE_KEYS, f'figures.{fid}')
        req = {'status', 'kind', 'name', 'xlabel', 'ylabel', 'series', 'order', 'scale'}
        if req - set(f):
            raise VisualizationError(f'Missing required registry keys for {fid}')
        if f['status'] not in {'draft', 'approved'} or f['kind'] not in {'ecdf', 'histogram', 'bootstrap_histogram', 'distribution_triptych', 'stacked_bar', 'dotplot', 'line_histogram'} or f['scale'] not in {'linear', 'log'}:
            raise VisualizationError(f'Invalid status/kind/scale for {fid}')
        if not all((isinstance(f[k], str) and f[k].strip() for k in ('name', 'xlabel', 'ylabel'))):
            raise VisualizationError(f'Invalid labels for {fid}')
        if f['series'] != f['order'] or not f['series'] or len(set(f['series'])) != len(f['series']) or set(f['series']) - set(r['semantics']):
            raise VisualizationError(f'Invalid series/order for {fid}')
        if 'bin_rule' in f and f['bin_rule'] != 'pooled_fd_sturges':
            raise VisualizationError(f'Invalid bin_rule for {fid}')
        if 'bins' in f and (isinstance(f['bins'], bool) or not isinstance(f['bins'], int) or f['bins'] < 1):
            raise VisualizationError(f'Invalid bins for {fid}')
        if 'xlim' in f and (not isinstance(f['xlim'], list) or len(f['xlim']) != 2 or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in f['xlim']))) or (f['xlim'][0] >= f['xlim'][1])):
            raise VisualizationError(f'Invalid xlim for {fid}')
        if f['scale'] == 'log' and 'xlim' in f and (f['xlim'][0] <= 0):
            raise VisualizationError(f'Log xlim must be positive for {fid}')
        if f['kind'] == 'ecdf':
            if 'ylim' not in f or not isinstance(f['ylim'], list) or len(f['ylim']) != 2 or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in f['ylim']))) or (f['ylim'][0] >= f['ylim'][1]):
                raise VisualizationError(f'ECDF ylim must be two ascending finite values for {fid}')
        elif f['kind'] in {'histogram', 'bootstrap_histogram'}:
            if not isinstance(f.get('density'), bool):
                raise VisualizationError(f'Histogram density must be boolean for {fid}')
            y = f.get('ylim')
            if not isinstance(y, list) or len(y) not in {1, 2} or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in y))) or (len(y) == 2 and y[0] >= y[1]):
                raise VisualizationError(f'Histogram ylim must be a finite bottom or ascending pair for {fid}')
            if f['kind'] == 'histogram' and (not ({'edges_start', 'edges_stop', 'edges_count'} <= set(f) or f.get('bin_rule') == 'pooled_fd_sturges')):
                raise VisualizationError(f'Histogram requires fixed edges or pooled bin rule for {fid}')
            if f['kind'] == 'bootstrap_histogram' and 'bins' not in f:
                raise VisualizationError(f'Bootstrap histogram requires bins for {fid}')
        edgekeys = {'edges_start', 'edges_stop', 'edges_count'} & set(f)
        if edgekeys and (edgekeys != {'edges_start', 'edges_stop', 'edges_count'} or not all((isinstance(f[k], (int, float)) and (not isinstance(f[k], bool)) and math.isfinite(f[k]) for k in ('edges_start', 'edges_stop'))) or f['edges_stop'] <= f['edges_start'] or isinstance(f['edges_count'], bool) or (not isinstance(f['edges_count'], int)) or (f['edges_count'] < 2)):
            raise VisualizationError(f'Invalid edge specification for {fid}')
    return Protocol(s, r, sha256(sr).hexdigest(), sha256(rr).hexdigest(), sp, rp)

def verify_style_integrity(p):
    if sha256(p.style_path.read_bytes()).hexdigest() != p.style_hash or sha256(p.registry_path.read_bytes()).hexdigest() != p.registry_hash:
        raise VisualizationError('Protocol files changed; reload')
    if tomllib.loads(p.style_path.read_text(encoding='utf-8')) != p.style or tomllib.loads(p.registry_path.read_text(encoding='utf-8')) != p.registry:
        raise VisualizationError('Protocol mappings were mutated; reload')

def require_arial():
    from matplotlib import font_manager
    if not any((f.name == 'Arial' for f in font_manager.fontManager.ttflist)):
        raise VisualizationError('Arial is not installed; font fallback is prohibited')

def apply_style(p):
    import matplotlib as mpl
    require_arial()
    s = p.style
    f = s['font']
    a = s['axes']
    c = s['color']
    rc = {'font.family': f['family'], 'font.weight': f['normal_weight'], 'font.size': f['tick_pt'], 'axes.labelsize': f['axis_label_pt'], 'axes.titlesize': f['panel_heading_pt'], 'axes.labelweight': f['normal_weight'], 'axes.titleweight': f['normal_weight'], 'xtick.labelsize': f['tick_pt'], 'ytick.labelsize': f['tick_pt'], 'legend.fontsize': f['legend_pt'], 'axes.linewidth': s['line']['axes_pt'], 'lines.linewidth': s['line']['data_pt'], 'lines.markersize': s['marker']['size_pt'], 'lines.markeredgewidth': s['marker']['edge_pt'], 'lines.markerfacecolor': s['marker']['face'], 'axes.edgecolor': c['foreground'], 'axes.labelcolor': c['foreground'], 'text.color': c['foreground'], 'xtick.color': c['foreground'], 'ytick.color': c['foreground'], 'figure.facecolor': c['background'], 'axes.facecolor': c['background'], 'axes.grid': a['grid'], 'xtick.direction': a['tick_direction'], 'ytick.direction': a['tick_direction'], 'xtick.top': False, 'ytick.right': False, 'xtick.minor.visible': False, 'ytick.minor.visible': False, 'xtick.major.size': a['major_length_pt'], 'ytick.major.size': a['major_length_pt'], 'xtick.major.width': a['major_width_pt'], 'ytick.major.width': a['major_width_pt'], 'xtick.major.pad': a['major_pad_pt'], 'ytick.major.pad': a['major_pad_pt'], 'axes.spines.left': True, 'axes.spines.right': True, 'axes.spines.top': True, 'axes.spines.bottom': True}
    rc.update(s['backend']['rcparams'])
    mpl.rcParams.update(rc)

def layout(n, p, *, columns=None, legend_rows=0, width_mm=None):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    if not isinstance(n, int) or not 1 <= n <= 26:
        raise VisualizationError('Panel count must be 1..26')
    l = p.style['layout']
    cols = columns if columns is not None else l['default_columns'][n - 1] if n <= 7 else 3
    if not isinstance(cols, int) or not 1 <= cols <= min(3, n):
        raise VisualizationError('columns must be 1..min(3,n)')
    if width_mm is not None:
        require_positive_style_value(width_mm, 'width_mm')
        width = float(width_mm)
    else:
        width = l['single_width_mm'] if cols == 1 else l['multi_width_mm']
    rows = (n + cols - 1) // cols
    pw = (width - l['left_mm'] - l['right_mm'] - (cols - 1) * l['hgap_mm']) / cols
    if pw <= 0:
        raise VisualizationError('width_mm is too small for physical margins')
    ph = pw / l['panel_aspect']
    leg = legend_rows * l['legend_row_mm'] + (l['legend_gap_mm'] if legend_rows else 0)
    height = l['bottom_mm'] + l['top_mm'] + leg + rows * ph + (rows - 1) * l['vgap_mm']
    fig = plt.figure(figsize=(width / MM_PER_INCH, height / MM_PER_INCH))
    axes = []
    a = p.style['axes']
    for i in range(n):
        row = i // cols
        nr = min(cols, n - row * cols)
        col = i % cols
        x = l['left_mm'] + (cols - nr) * (pw + l['hgap_mm']) / 2 + col * (pw + l['hgap_mm'])
        y = l['bottom_mm'] + (rows - 1 - row) * (ph + l['vgap_mm'])
        ax = fig.add_axes([x / width, y / height, pw / width, ph / height])
        axes.append(ax)
        ax.tick_params(top=False, right=False, bottom=True, left=True, direction='in', length=a['major_length_pt'], width=a['major_width_pt'], pad=a['major_pad_pt'])
        ax.minorticks_off()
        ax.grid(False)
        ax.xaxis.set_major_locator(MaxNLocator(a['major_target']))
        ax.yaxis.set_major_locator(MaxNLocator(a['major_target']))
        ax.text(a['panel_label_x_mm'] / pw, 1 + a['panel_label_y_mm'] / ph, chr(97 + i), transform=ax.transAxes, fontsize=p.style['font']['panel_letter_pt'], fontweight=p.style['font']['panel_letter_weight'], va='bottom')
    return (fig, axes)

def finite(values, *, policy='raise'):
    if policy not in {'raise', 'drop'}:
        raise VisualizationError('finite_policy must be raise or drop')
    a = np.asarray(values, float)
    m = np.isfinite(a)
    if not m.all() and policy == 'raise':
        raise VisualizationError('Non-finite values present; select drop explicitly')
    a = a[m]
    if not len(a):
        raise VisualizationError('No finite observations')
    return a

def ecdf(values, *, finite_policy='raise'):
    x = np.sort(finite(values, policy=finite_policy), kind='mergesort')
    return (x, np.arange(1, len(x) + 1) / len(x))

def pooled_edges(series, *, finite_policy='raise'):
    x = np.concatenate([finite(v, policy=finite_policy) for v in series])
    if x.min() == x.max():
        return np.array([x[0] - 0.5, x[0] + 0.5])
    q = np.percentile(x, [25, 75])
    iqr = q[1] - q[0]
    bins = int(np.ceil((x.max() - x.min()) / (2 * iqr / np.cbrt(len(x))))) if iqr > 0 else int(np.ceil(np.log2(len(x))) + 1)
    return np.linspace(x.min(), x.max(), max(1, bins) + 1)

def _record(fig, d):
    fig._phonetics_resolution = d

def plot_ecdf(ax, series, figure, p, *, finite_policy='raise'):
    counts = {}
    for sid in figure['order']:
        x, y = ecdf(series[sid], finite_policy=finite_policy)
        counts[sid] = len(x)
        sem = p.registry['semantics'][sid]
        ax.step(x, y, where='post', label=sem['label'], color=sem['color'], linestyle=sem['linestyle'], marker=None, linewidth=p.style['line']['data_pt'])
    ax.set(xlabel=figure['xlabel'], ylabel=figure['ylabel'], ylim=tuple(figure['ylim']), xscale=figure['scale'])
    ax.set_xlim(tuple(figure['xlim'])) if 'xlim' in figure else None
    _record(ax.figure, {'kind': 'ecdf', 'finite_policy': finite_policy, 'counts': counts, 'limits': {'x': list(ax.get_xlim()), 'y': list(ax.get_ylim())}})

def plot_histogram(ax, series, figure, p, *, edges=None, finite_policy='raise'):
    arr = [finite(series[s], policy=finite_policy) for s in figure['order']]
    edges = np.linspace(figure['edges_start'], figure['edges_stop'], figure['edges_count']) if edges is None and 'edges_count' in figure else edges
    edges = np.histogram_bin_edges(np.concatenate(arr), bins=figure['bins']) if edges is None and 'bins' in figure else pooled_edges(arr) if edges is None else np.asarray(edges, float)
    edges = np.asarray(edges, float)
    if len(edges) < 2 or not np.all(np.isfinite(edges)) or (not np.all(np.diff(edges) > 0)):
        raise VisualizationError('Histogram edges must be finite and strictly increasing')
    for sid, x in zip(figure['order'], arr):
        sem = p.registry['semantics'][sid]
        ax.hist(x, bins=edges, density=figure['density'], histtype='step', label=sem['label'], color=sem['color'], linestyle=sem['linestyle'], linewidth=p.style['line']['data_pt'])
    ax.set(xlabel=figure['xlabel'], ylabel=figure['ylabel'], xscale=figure['scale'])
    ax.set_ylim(tuple(figure['ylim'])) if len(figure['ylim']) == 2 else ax.set_ylim(bottom=figure['ylim'][0])
    ax.set_xlim(tuple(figure['xlim'])) if 'xlim' in figure else None
    _record(ax.figure, {'kind': figure['kind'], 'histogram_edges': edges.tolist(), 'finite_policy': finite_policy, 'counts': {s: len(x) for s, x in zip(figure['order'], arr)}, 'limits': {'x': list(ax.get_xlim()), 'y': list(ax.get_ylim())}})
    return edges

def shared_legend(fig, axes, p, *, rows=1):
    h, l = axes[0].get_legend_handles_labels()
    c = p.style['legend']
    return fig.legend(h, l, loc=c['location'], ncol=min(c['max_per_row'], max(1, int(np.ceil(len(l) / rows)))), frameon=False, handlelength=c['handle_length_em'], columnspacing=c['column_spacing_em'])

def export_figure(fig, output_dir, stem, figure_id, p, *, scientific_provenance, overwrite=None):
    verify_style_integrity(p)
    spec = p.figure(figure_id)
    if spec['status'] != 'approved':
        raise VisualizationError(f'Figure {figure_id!r} is draft; approval required')
    if not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]*', stem) or stem in {'.', '..'}:
        raise VisualizationError('Unsafe export stem')
    prov = dict(scientific_provenance)
    if not re.fullmatch('[0-9a-fA-F]{64}', str(prov.get('source_digest', ''))):
        raise VisualizationError('source_digest must be SHA-256 hex')
    require_arial()
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    if not out.is_dir():
        raise VisualizationError('Export target is not a directory')
    paths = [out / f'{stem}.{x}' for x in p.style['export']['formats']]
    mp = out / f'{stem}.metadata.json'
    allow = p.style['export']['overwrite'] if overwrite is None else overwrite
    if not allow and any((x.exists() for x in paths + [mp])):
        raise FileExistsError('Refusing to overwrite export or metadata')
    from matplotlib.colors import to_rgba_array
    for a in fig.findobj():
        if hasattr(a, 'get_visible') and (not a.get_visible()):
            continue
        alpha = a.get_alpha() if hasattr(a, 'get_alpha') else None
        if alpha is not None and 0 < alpha < 1:
            raise VisualizationError('EPS forbids partial transparency')
        for getter in ('get_color', 'get_markerfacecolor', 'get_markeredgecolor', 'get_facecolor', 'get_edgecolor'):
            if not hasattr(a, getter):
                continue
            try:
                rgba = to_rgba_array(getattr(a, getter)())
            except (TypeError, ValueError):
                continue
            if rgba.size and np.any((rgba[:, 3] > 0) & (rgba[:, 3] < 1)):
                raise VisualizationError('EPS forbids RGBA partial transparency')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    box = fig.get_window_extent(renderer)
    from matplotlib.legend import Legend
    from matplotlib.text import Text
    ticks = set()
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            lo, hi = sorted(axis.get_view_interval())
            tol = max(1, abs(hi - lo)) * 1e-12
            for tick in axis._update_ticks():
                loc = tick.get_loc()
                if math.isfinite(loc) and lo - tol <= loc <= hi + tol:
                    ticks.update((x for x in (tick.label1, tick.label2) if x.get_visible()))
    for a in fig.findobj(lambda x: isinstance(x, (Text, Legend))):
        if isinstance(a, Text) and a not in ticks and any((a is z for ax in fig.axes for z in ax.get_xticklabels() + ax.get_yticklabels())):
            continue
        if not a.get_visible():
            continue
        b = a.get_window_extent(renderer)
        if b.width and b.height and (b.x0 < box.x0 - 1 or b.y0 < box.y0 - 1 or b.x1 > box.x1 + 1 or (b.y1 > box.y1 + 1)):
            raise VisualizationError('Visible text or legend is clipped')
    meta = {'schema_version': 1, 'config_schema_version': p.style['schema']['version'], 'figure_id': figure_id, 'style_hash': p.style_hash, 'registry_hash': p.registry_hash, 'resolved_style': p.style, 'resolved_figure': spec, 'resolved_data': getattr(fig, '_phonetics_resolution', {}), 'scientific_provenance': prov}
    temps = []
    try:
        for dest in paths:
            t = dest.with_name(dest.stem + '.tmp' + dest.suffix)
            temps.append(t)
            import matplotlib as mpl
            with mpl.rc_context({'savefig.bbox': None, 'savefig.pad_inches': 0, 'savefig.dpi': p.style['export']['dpi'], 'savefig.facecolor': p.style['color']['background'], 'savefig.transparent': False}):
                fig.savefig(t, format=dest.suffix[1:], dpi=p.style['export']['dpi'], bbox_inches=None, pad_inches=0, transparent=False, facecolor=p.style['color']['background'], metadata={'Creator': 'phonetics.visualization'})
        mt = mp.with_name(mp.stem + '.tmp' + mp.suffix)
        temps.append(mt)
        mt.write_text(json.dumps(meta, indent=2, sort_keys=True))
        for t, d in zip(temps, paths + [mp]):
            os.replace(t, d)
    finally:
        for t in temps:
            if t.exists():
                t.unlink()
    return paths


### Forest, geometry and residual Q–Q panel definitions


In [ ]:
def _grow_height(fig, axes, panel_mm):
    """Grow categorical panels to fit their rows using physical font spacing."""
    width, old_height = fig.get_size_inches() * 25.4
    current = axes[0].get_position().height * old_height
    growth = max(0.0, panel_mm - current)
    height = old_height + growth
    positions = [ax.get_position().bounds for ax in axes]
    fig.set_size_inches(width / 25.4, height / 25.4)
    for ax, (x, y, w, h) in zip(axes, positions):
        ax.set_position([x, y * old_height / height, w, (h * old_height + growth) / height])

def build_anchor_forest(rows, protocol):
    """Three component panels; each transition has five separate model rows."""
    p = protocol
    spec = p.figure('anchor_q_transitions')
    models = spec['order']
    fig, axes = layout(3, p, columns=3, legend_rows=2, width_mm=180)
    try:
        _grow_height(fig, axes, p.style['font']['tick_pt'] * 25.4 / 72 * 1.3 * 27)
        for ax in axes:
            pos = ax.get_position()
            ax.set_position([pos.x0 + 0.055, pos.y0, pos.width - 0.055, pos.height])
        found = {}
        for row in rows:
            key = (row['model_id'], row['edge_id'])
            if key in found:
                raise ValueError('duplicate forest model/transition')
            found[key] = row
        values = []
        for ax, comp in zip(axes, ('initial', 'final', 'tone')):
            edges = [r['edge_id'] for r in rows if r['model_id'] == models[0] and r['edge_id'].startswith(comp + ':')]
            if len(edges) != 4 or sum((e.endswith('->111') for e in edges)) != 1:
                raise ValueError('four transitions per component are required')
            edges = [e for e in edges if not e.endswith('->111')] + [e for e in edges if e.endswith('->111')]
            centers = [0, 7, 14, 22]
            for block, edge in enumerate(edges):
                for mi, model in enumerate(models):
                    row = found[model, edge]
                    seed_rows = row['seed_points']
                    if {s['seed'] for s in seed_rows} != set(range(5)) or len(seed_rows) != 5:
                        raise ValueError('five forest training seeds are required')
                    sd = row.get('sample_sd')
                    if sd is None:
                        sd = float(np.std([s['q'] for s in seed_rows], ddof=1))
                    mean = row['mean']
                    if not np.isfinite([mean, sd]).all() or sd < 0:
                        raise ValueError('invalid forest estimate')
                    values.extend((mean - sd, mean + sd))
                    sem = p.registry['semantics'][model]
                    ax.errorbar(mean, centers[block] + (mi - 2) * 0.9, xerr=sd, fmt=sem['marker'], color=sem['color'], mfc='white', ms=p.style['marker']['size_pt'], lw=p.style['line']['interval_pt'], capsize=p.style['line']['caps_pt'], label=sem['label'] if block == 0 else None)
            ax.axvline(0.5, color=p.style['color']['secondary'], ls='--', lw=p.style['line']['reference_pt'])
            ax.axhline(18.4, color=p.style['color']['secondary'], lw=p.style['line']['reference_pt'])
            ax.set(yticks=centers, yticklabels=[e.split(':')[1].replace('->', ' → ') for e in edges], ylim=(25.5, -4), title=comp.title(), xlabel='Q')
            ax.tick_params(axis='y', length=0)
        if len(found) != 60:
            raise ValueError('exact 12 transitions × five models required')
        lo, hi = (min(0.4, min(values) - 0.015), max(1.0, max(values) + 0.015))
        for ax in axes:
            ax.set_xlim(lo, hi)
        shared_legend(fig, axes, p, rows=2)
        fig._phonetics_resolution = {'kind': 'forest', 'interval': 'sample SD of five training seeds', 'reference_q': 0.5, 'full_match': 'separated final block', 'mc_se_in_figure': False, 'xlim': [lo, hi], 'model_order': models}
        return fig
    except BaseException:
        import matplotlib.pyplot as plt
        plt.close(fig)
        raise

def build_dimsim_geometry(plot, protocol):
    """Common data-covering cosine range, per-panel counts and support gaps."""
    p = protocol
    models = p.figure('dimsim_geometry')['order']
    curves = plot['curves']
    threshold = plot.get('min_unique_pairs_per_bin', 100)
    if type(threshold) is not int or threshold <= 0:
        raise ValueError('invalid unique-pair support threshold')
    values = [v for row in curves for v in [row['mean_cosine'], *[s['mean_cosine'] for s in row['seed_points']]] if v is not None]
    if not values or not np.isfinite(values).all():
        raise ValueError('finite geometry means and seed curves required')
    span = max(max(values) - min(values), 0.1)
    limits = (min(values) - span * 0.05, max(values) + span * 0.05)
    max_count = max((s['draw_count'] for r in curves for s in r['seed_points']))
    locator = MaxNLocator(nbins=5, integer=True, min_n_ticks=3)
    ticks = locator.tick_values(0, max(1, max_count))
    ticks = ticks[ticks >= 0]
    count_top = float(ticks[-1])
    fig, axes = layout(10, p, columns=2, legend_rows=2, width_mm=180)
    try:
        for host in axes:
            pos = host.get_position()
            host.set_position([pos.x0, pos.y0, pos.width - 0.06, pos.height])
        for mi, model in enumerate(models):
            for col, mode in enumerate(('dimsim-natural', 'dimsim-balanced')):
                ax = axes[2 * mi + col]
                rows = [r for r in curves if r['model_id'] == model and r['mode'] == mode]
                if len(rows) != 13:
                    raise ValueError('all 13 DIMSIM bins required')
                if len({r['bin'] for r in rows}) != 13:
                    raise ValueError('duplicate DIMSIM bins')
                cx = ax.twinx()
                ax.set_zorder(cx.get_zorder() + 1)
                ax.patch.set_visible(False)
                draws = [r['seed_points'][0]['draw_count'] for r in rows]
                unique = [r['seed_points'][0]['unique_pair_count'] for r in rows]
                for r in rows:
                    if {s['seed'] for s in r['seed_points']} != set(range(5)) or len(r['seed_points']) != 5:
                        raise ValueError('five geometry training seeds required')
                    if any(((s['draw_count'], s['unique_pair_count']) != (r['seed_points'][0]['draw_count'], r['seed_points'][0]['unique_pair_count']) for s in r['seed_points'])):
                        raise ValueError('geometry counts differ across seeds')
                cx.bar(range(13), draws, width=0.76, color='#e6e6e6', edgecolor='none')
                cx.bar(range(13), unique, width=0.76, facecolor='none', edgecolor='#555555', linewidth=p.style['axes']['major_width_pt'])
                for si in range(5):
                    y = [next((s['mean_cosine'] for s in r['seed_points'] if s['seed'] == si)) for r in rows]
                    for j in range(12):
                        if rows[j]['supported'] and rows[j + 1]['supported'] and (y[j] is not None) and (y[j + 1] is not None):
                            ax.plot([j, j + 1], y[j:j + 2], color='#aaaaaa', lw=p.style['line']['reference_pt'])
                    missing = [(j, v) for j, v in enumerate(y) if not rows[j]['supported'] and v is not None]
                    if missing:
                        x, vals = zip(*missing)
                        ax.plot(x, vals, ls='none', marker='o', mfc='white', mec='#aaaaaa', ms=p.style['marker']['size_pt'] * 0.65)
                for j, row in enumerate(rows):
                    y = row['mean_cosine']
                    if y is not None:
                        ax.plot([j], [y], ls='none', marker='o', mfc='#171717' if row['supported'] else 'white', mec='#171717', ms=p.style['marker']['size_pt'])
                    if j < 12 and row['supported'] and rows[j + 1]['supported'] and (y is not None) and (rows[j + 1]['mean_cosine'] is not None):
                        ax.plot([j, j + 1], [y, rows[j + 1]['mean_cosine']], color='#171717', lw=p.style['line']['data_pt'])
                cx.set(ylim=(0, count_top), yticks=ticks, ylabel='Count')
                cx.tick_params(labelsize=p.style['font']['tick_pt'], direction='in')
                ax.set(ylim=limits, xlim=(-0.5, 12.5), xticks=[0, 4, 8, 12], xticklabels=['5', '45', '85', '120+'], ylabel='Cosine similarity' if col == 0 else '', title=p.registry['semantics'][model]['label'] + ' · ' + ('Natural' if col == 0 else 'Balanced'))
                if mi == 4:
                    ax.set_xlabel('DIMSIM distance')
        handles = [Line2D([], [], color='#171717', label='Mean cosine'), Line2D([], [], color='#aaaaaa', lw=p.style['line']['reference_pt'], label='Seed curves'), Line2D([], [], color='#171717', ls='none', marker='o', mfc='white', label=f'Support < {threshold}'), Patch(facecolor='#e6e6e6', label='All draws'), Patch(facecolor='none', edgecolor='#555555', label='Unique pairs')]
        fig.legend(handles=handles, loc='upper center', ncol=3, frameon=False, handlelength=p.style['legend']['handle_length_em'])
        fig._phonetics_resolution = {'kind': 'dimsim_geometry', 'cosine_limits': list(limits), 'counts_limits': [0, count_top], 'counts_ticks': ticks.tolist(), 'boundary': 'removed', 'support_gaps_not_bridged': True, 'counts_encoding': 'light fill and dark outline', 'natural_counts': 'draws and unique pairs coincide by design', 'balanced_counts': 'repeated draws retained', 'support_threshold': threshold}
        return fig
    except BaseException:
        import matplotlib.pyplot as plt
        plt.close(fig)
        raise

def build_residual_qq(qq_data, protocol):
    """Render prepared quantiles only, without standardization or model fitting."""
    p = protocol
    spec = p.figure('residual_qq')
    fig, axes = layout(6, p, columns=2, legend_rows=1, width_mm=180)
    try:
        curves = qq_data['curves']
        valid = [r for r in curves if r['status'] == 'ok']
        x = [v for r in valid for v in r['normal_quantiles']]
        y = [v for r in valid for v in r['standardized_residuals']]
        xlim = (min(x) - 0.1, max(x) + 0.1) if x else (-1.0, 1.0)
        ylim = (min(y) - 0.1, max(y) + 0.1) if y else (-1.0, 1.0)
        diagnostics = []
        for ax, model in zip(axes, spec['order']):
            rows = [r for r in curves if r['model_id'] == model]
            expected = {None} if model == 'dimsim' else set(range(5))
            if len(rows) != len(expected) or {r['seed'] for r in rows} != expected:
                raise ValueError('invalid Q–Q model/seed curve inventory')
            for row in rows:
                if row['status'] != 'ok':
                    diagnostics.append({'model_id': model, 'seed': row['seed'], 'status': row['status']})
                    continue
                xx, yy = (row['normal_quantiles'], row['standardized_residuals'])
                if len(xx) != len(yy) or not len(xx) or (not np.isfinite(xx + yy).all()):
                    raise ValueError('invalid Q–Q coordinates')
                ax.plot(xx, yy, color='#171717' if model == 'dimsim' else '#777777', lw=p.style['line']['reference_pt'])
            ax.plot(xlim, xlim, color='#999999', ls='--', lw=p.style['line']['reference_pt'])
            if not any((r['status'] == 'ok' for r in rows)):
                ax.text(0.5, 0.5, 'Undefined', transform=ax.transAxes, ha='center')
            ax.set(xlim=xlim, ylim=ylim, xlabel=spec['xlabel'], ylabel=spec['ylabel'], title=p.registry['semantics'][model]['label'])
        fig.legend(handles=[Line2D([], [], color='#777777', label='Training seeds'), Line2D([], [], color='#171717', label='DIMSIM'), Line2D([], [], color='#999999', ls='--', label='Normal reference')], loc='upper center', ncol=3, frameon=False)
        fig._phonetics_resolution = {'kind': 'residual_qq', 'standardization': qq_data['config'], 'xlim': list(xlim), 'ylim': list(ylim), 'undefined': diagnostics, 'normality_test': False}
        return fig
    except BaseException:
        import matplotlib.pyplot as plt
        plt.close(fig)
        raise


### Five evaluation figures


In [ ]:
MODEL_LABELS = {'syllable': 'Syllable', 'components': 'Components', 'acoustic': 'Acoustic', 'components-acoustic-lambda-1': 'Combined λ1', 'components-acoustic-lambda-10': 'Combined λ10'}
FIGURE_STEMS = ('anchor-q-transitions', 'transition-shares', 'component-shares', 'dimsim-geometry', 'dimsim-correlation')

def model_semantics(p, model):
    return p.registry['semantics'][model]

def manifest_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_plot_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def guard_report_output(target):
    path = Path(target).resolve()
    for ancestor in (path, *path.parents):
        manifest = ancestor / 'manifest.json'
        if not manifest.is_file():
            continue
        try:
            data = read_plot_json(manifest)
        except (OSError, ValueError):
            continue
        if str(data.get('analysis', '')).startswith('contextual-') and data.get('status') in {'complete', 'failed'}:
            raise ValueError(f'Refusing to write inside immutable evaluation run: {ancestor}')

def check_paired_outputs(aq, ds):
    am, dm = (aq['manifest'], ds['manifest'])
    if am.get('analysis') != 'contextual-anchor-q' or dm.get('analysis') != 'contextual-dimsim':
        raise ValueError('output roles must be anchor-Q and DIMSIM respectively')
    ap, dp = (am.get('scientific_input_pins', {}), dm.get('scientific_input_pins', {}))
    if ap.get('analysis') != 'contextual-anchor-q' or dp.get('analysis') != 'contextual-dimsim':
        raise ValueError('scientific output roles do not match manifests')
    for key in ('bundle_manifest_sha256', 'model_ids', 'seeds'):
        if ap.get(key) != dp.get(key):
            raise ValueError(f'paired outputs differ in {key}')

    def checkpoints(pins):
        rows = pins.get('model_seed_provenance')
        if not isinstance(rows, list):
            raise ValueError('model/seed checkpoint provenance is required')
        result = {(x.get('model_id'), x.get('seed')): x for x in rows if isinstance(x, dict)}
        if len(result) != len(rows) or not result or any((not re.fullmatch('[0-9a-f]{64}', str(x.get('checkpoint_sha256', ''))) for x in result.values())):
            raise ValueError('invalid model/seed checkpoint provenance')
        return result
    if checkpoints(ap) != checkpoints(dp):
        raise ValueError('paired outputs use different model/seed checkpoint provenance')

def build_figures(anchor_plot, dimsim_plot, protocol=None):
    """Build figures from the two persisted report plot_data objects."""
    p = protocol or load_protocol()
    figs = []
    models = list(MODEL_LABELS)
    try:
        components = ('initial', 'final', 'tone')
        fig = build_anchor_forest(anchor_plot['edges'], p)
        figs.append(fig)
        fig, axs = layout(1, p, columns=1, legend_rows=2, width_mm=180)
        figs.append(fig)
        ax = axs[0]
        key, field, n = ('transition_shares', 'edge_id', 12)
        rs = anchor_plot[key]
        for ax in axs:
            for ci, comp in enumerate(components):
                values = []
            first_valid = next((m for m in models if all((r['share'] is not None for r in rs if r['model_id'] == m))), None)
            for mi, model in enumerate(models):
                names = [r[field] for r in rs if r['model_id'] == model]
                vals = [next((r['share'] for r in rs if r['model_id'] == model and r[field] == name)) for name in names]
                if len(vals) != n or any((v is None for v in vals)):
                    continue
                if n == 12:
                    bottom = 0.0
                    for ei, (name, val) in enumerate(zip(names, vals)):
                        component = name.split(':', 1)[0]
                        shade = ('#555555', '#999999', '#cccccc')[('initial', 'final', 'tone').index(component)]
                        hatch = ('', '/', '\\', 'x')[ei % 4]
                        ax.bar(mi, val, bottom=bottom, width=0.68, color=shade, hatch=hatch, edgecolor='black', linewidth=0.25, label=f"{component.title()} {name.split(':', 1)[1]}" if model == first_valid else None)
                        bottom += val
                else:
                    b = 0
                    for ci, val in enumerate(vals):
                        ax.bar(mi, val, bottom=b, width=0.65, color=('#444444', '#888888', '#c5c5c5')[ci], hatch=('', '//', 'xx')[ci], edgecolor='black', linewidth=0.3, label=components[ci] if mi == 0 else None)
                        b += val
            ax.set_ylim(0, 1)
            ax.set_ylabel('Q share')
            ax.set_xticks(np.arange(5))
            ax.set_xticklabels([MODEL_LABELS[m] for m in models], rotation=25, ha='right')
            if n == 12:
                for mi, model in enumerate(models):
                    if any((r['model_id'] == model and r.get('status') == 'undefined_zero_total' for r in rs)):
                        ax.text(mi, 0.5, 'undefined', rotation=90, ha='center', va='center')
        shared_legend(fig, axs, p, rows=2)
        fig._phonetics_resolution = {'kind': 'anchor_q_transition_shares', 'undefined_zero_total': 'shown as absent bars'}
        fig, axs = layout(1, p, columns=1, legend_rows=1, width_mm=180)
        figs.append(fig)
        ax = axs[0]
        rs = anchor_plot['component_shares']
        first_component_valid = next((m for m in models if all((r['share'] is not None for r in rs if r['model_id'] == m))), None)
        for mi, model in enumerate(models):
            vals = [next((r['share'] for r in rs if r['model_id'] == model and r['component'] == comp)) for comp in components]
            if any((v is None for v in vals)):
                ax.text(mi, 0.5, 'undefined', rotation=90, ha='center', va='center')
                continue
            bottom = 0.0
            for ci, (comp, val) in enumerate(zip(components, vals)):
                ax.bar(mi, val, bottom=bottom, width=0.65, color=('#444444', '#888888', '#c5c5c5')[ci], hatch=('', '//', 'xx')[ci], edgecolor='black', linewidth=0.3, label=comp.title() if model == first_component_valid else None)
                bottom += val
        ax.set(ylim=(0, 1), ylabel='Q share', xticks=range(5), xticklabels=[MODEL_LABELS[m] for m in models])
        shared_legend(fig, axs, p)
        fig._phonetics_resolution = {'kind': 'anchor_q_component_shares', 'undefined_zero_total': 'shown as absent bars'}
        fig = build_dimsim_geometry(dimsim_plot, p)
        figs.append(fig)
        fig, axs = layout(2, p, columns=2, legend_rows=2, width_mm=180)
        figs.append(fig)
        rho = dimsim_plot.get('rho', [])
        for ax, mode in zip(axs, ('dimsim-natural', 'dimsim-balanced')):
            rr = [r for r in rho if r['mode'] == mode]
            for mi, model in enumerate(models):
                r = next((v for v in rr if v['model_id'] == model), None)
                if not r or r.get('mean') is None:
                    continue
                s = model_semantics(p, model)
                ax.scatter([mi], [r['mean']], marker=s['marker'], c=s['color'], label=MODEL_LABELS[model])
                vals = [x['rho'] for x in r['seed_values'] if x['rho'] is not None]
                ax.scatter([mi] * len(vals), vals, marker='.', c='#aaaaaa', s=10)
            ax.axhline(0, color='#777777', lw=p.style['line']['reference_pt'])
            ax.set(title=mode, xlim=(-0.5, 4.5), xticks=range(5), xticklabels=[MODEL_LABELS[m] for m in models], ylabel='Spearman ρ')
            ax.tick_params(axis='x', labelrotation=25)
        shared_legend(fig, axs, p, rows=2)
        fig._phonetics_resolution = {'kind': 'dimsim_correlation', 'five_seed_points': True}
        return figs
    except Exception:
        for fig in figs:
            plt.close(fig)
        raise

def render_evaluation(anchor_output, dimsim_output, output_dir, source_digest, *, protocol=None, residual_qq=None):
    """Render five PNG/EPS pairs after validating both complete output manifests."""
    aq = load_evaluation_outputs(anchor_output)
    ds = load_evaluation_outputs(dimsim_output)
    return render_loaded_evaluation(aq, ds, output_dir, source_digest, protocol=protocol, residual_qq=residual_qq)

def render_loaded_evaluation(aq, ds, output_dir, source_digest, *, protocol=None, residual_qq=None):
    """Render already validated inputs without reloading large raw batches."""
    check_paired_outputs(aq, ds)
    if not isinstance(source_digest, str) or not re.fullmatch('[0-9a-f]{64}', source_digest):
        raise ValueError('source_digest must be an explicit lowercase SHA-256 digest')
    p = protocol or load_protocol()
    apply_style(p)
    out = Path(output_dir)
    guard_report_output(out)
    stems = FIGURE_STEMS + (('residual-qq',) if residual_qq is not None else ())
    targets = [out / f'{stem}.{ext}' for stem in stems for ext in ('png', 'eps', 'metadata.json')]
    if any((x.exists() for x in targets)):
        raise FileExistsError('Refusing to overwrite evaluation figure exports')
    dplot = dict(ds['plot_data'])
    dplot['rho'] = ds['tables'].get('model_rho', [])
    dplot['min_unique_pairs_per_bin'] = ds.get('report', {}).get('contract', {}).get('analysis', {}).get('min_unique_pairs_per_bin', 100)
    aplot = dict(aq['plot_data'])
    dispersions = {(r['model_id'], r['edge_id']): r['sample_sd'] for r in aq['tables'].get('model_edge_seed_summary', [])}
    aplot['edges'] = [{**r, 'sample_sd': dispersions[r['model_id'], r['edge_id']]} if (r['model_id'], r['edge_id']) in dispersions else r for r in aplot['edges']]
    figs = build_figures(aplot, dplot, p)
    saved = []
    try:
        if residual_qq is not None:
            figs.append(build_residual_qq(residual_qq, p))
        for fig, stem, fid in zip(figs, stems, ('anchor_q_transitions', 'transition_shares', 'component_shares', 'dimsim_geometry', 'dimsim_correlation', 'residual_qq')):
            saved.extend(export_figure(fig, out, stem, fid, p, scientific_provenance={'source_digest': source_digest, 'anchor_q_output': str(aq['root']), 'dimsim_output': str(ds['root']), 'anchor_q_manifest_sha256': manifest_digest(aq['root'] / 'manifest.json'), 'dimsim_manifest_sha256': manifest_digest(ds['root'] / 'manifest.json')}))
            saved.append(out / f'{stem}.metadata.json')
        return saved
    finally:
        for fig in figs:
            plt.close(fig)


### Verified interpretation presentation data


In [ ]:
def load_interpretation_presentation_data(root):
    root = Path(root).resolve(strict=True)
    names = ['coefficients.md', 'anchor-q-groups.md', 'anchor-q-group-differences.md']
    names += [f'top-50-{m}.md' for m in (*MODEL_LABELS, 'dimsim')]
    expected = {'config.json', 'data/presentation_data.json', 'data/top_pairs.json', 'diagnostics/summary.json'}
    expected.update(('tables/' + name for name in names))
    manifest = verify_artifact_inventory(root, 'contextual-interpretation-data', root.name, expected)
    data = read_finite_json(root / 'data/presentation_data.json')
    if data.get('schema_version') != 1 or set(data.get('tables', {})) != set(names) or data.get('bundle_manifest_sha256') != manifest['scientific_input_pins'].get('evaluation_bundle_manifest_sha256') or (data.get('provenance') != manifest['scientific_input_pins'].get('model_seed_provenance')):
        raise ValueError('interpretation presentation identity mismatch')
    for name, body in data['tables'].items():
        if (root / 'tables' / name).read_text(encoding='utf-8') != body:
            raise ValueError('readable tables disagree with prepared data')
    if read_finite_json(root / 'data/top_pairs.json') != data['top_pairs']:
        raise ValueError('top-pair identities disagree with prepared data')
    return (data, manifest)


### Presentation run


In [ ]:
def write_evaluation_report(project_root, run_id, *, anchor_q_output, dimsim_output, random_control_output, source_hashes, source_digest, resolved_config=None, protocol=None, interpretation_data_output=None):
    """Save tables and PNG/EPS figures without checkpoints, extraction or scoring."""
    run_id = safe_run_id(run_id)
    if not isinstance(source_digest, str) or re.fullmatch('[0-9a-f]{64}', source_digest) is None:
        raise ValueError('source_digest must be an explicit lowercase SHA-256 digest')
    inputs = [Path(path).resolve(strict=True) for path in (anchor_q_output, dimsim_output, random_control_output)]
    target = Path(project_root).resolve() / 'outputs' / 'contextual-evaluation-report' / run_id
    ensure_output_outside_inputs(target, inputs)
    if target.exists():
        raise FileExistsError(f'refusing to overwrite existing output directory: {target}')
    guard_report_output(target)
    if not isinstance(source_hashes, dict) or not source_hashes or any((not isinstance(value, str) or re.fullmatch('[0-9a-f]{64}', value) is None for value in source_hashes.values())):
        raise ValueError('explicit source code SHA-256 map is required')
    if resolved_config is not None and (not isinstance(resolved_config, dict)):
        raise ValueError('resolved_config must be a mapping')
    aq, ds = (load_evaluation_summary(inputs[0]), load_evaluation_summary(inputs[1]))
    check_paired_outputs(aq, ds)
    random = load_random_control_summary(inputs[2])
    prepared = None
    if interpretation_data_output is not None:
        interpretation_data_output = Path(interpretation_data_output).resolve(strict=True)
        prepared, prepared_manifest = load_interpretation_presentation_data(interpretation_data_output)
        pins = aq['manifest']['scientific_input_pins']
        expected_provenance = {(r['model_id'], r['seed']): {k: r[k] for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256')} for r in pins['model_seed_provenance']}
        actual_provenance = {(m, r['seed']): {k: r[k] for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256')} for m, rows in prepared['provenance'].items() for r in rows}
        if prepared['bundle_manifest_sha256'] != pins['bundle_manifest_sha256'] or actual_provenance != expected_provenance:
            raise ValueError('interpretation and evaluation use different inputs')
        inputs.append(interpretation_data_output)
    bundle_hash = aq['manifest']['scientific_input_pins']['bundle_manifest_sha256']
    if random['manifest']['scientific_input_pins']['bundle_manifest_sha256'] != bundle_hash:
        raise ValueError('random control and learned models use different evaluation bundles')
    p = protocol or load_protocol()
    apply_style(p)
    config = {**(resolved_config or {}), 'anchor_q_output': str(inputs[0]), 'dimsim_output': str(inputs[1]), 'random_control_output': str(inputs[2]), 'source_digest': source_digest, 'style_sha256': p.style_hash, 'registry_sha256': p.registry_hash, 'formats': list(p.style['export']['formats']), 'png_dpi': p.style['export']['dpi'], 'interpretation_data_output': str(interpretation_data_output) if prepared is not None else None}
    pins = {'evaluation_bundle_manifest_sha256': bundle_hash, 'anchor_q_manifest_sha256': file_sha256(inputs[0] / 'manifest.json'), 'dimsim_manifest_sha256': file_sha256(inputs[1] / 'manifest.json'), 'random_control_manifest_sha256': file_sha256(inputs[2] / 'manifest.json'), 'model_ids': aq['manifest']['scientific_input_pins']['model_ids'], 'seeds': aq['manifest']['scientific_input_pins']['seeds']}
    with tempfile.TemporaryDirectory(prefix='evaluation-tables-') as directory:
        stage = Path(directory)
        table_paths = render_evaluation_tables(aq['tables'], ds['tables'], stage / 'tables')
        if prepared is not None:
            for name, body in prepared['tables'].items():
                path = stage / 'tables' / name
                with path.open('x', encoding='utf-8', newline='\n') as stream:
                    stream.write(body)
                table_paths.append(path)
        stems = FIGURE_STEMS + (('residual-qq',) if prepared is not None else ())
        expected = {'config.json', 'data/random_control_summary.json', 'diagnostics/summary.json'}
        if prepared is not None:
            expected.add('data/interpretation_summary.json')
        expected.update((f'tables/{path.name}' for path in table_paths))
        expected.update((f'figures/{stem}.{suffix}' for stem in stems for suffix in ('png', 'eps', 'metadata.json')))
        with presentation_run(project_root, 'contextual-evaluation-report', run_id, input_roots=inputs, config=config, source_hashes=source_hashes, expected_files=expected, scientific_pins=pins, profile='contextual_evaluation_report', figure_exports_on_complete='complete') as root:
            for path in table_paths:
                shutil.copyfile(path, root / 'tables' / path.name)
            write_data_json(root / 'data/random_control_summary.json', random['report'])
            if prepared is not None:
                write_data_json(root / 'data/interpretation_summary.json', {k: prepared[k] for k in ('q_groups', 'regression_summary', 'regression_per_seed', 'top_pairs')})
            render_loaded_evaluation(aq, ds, root / 'figures', source_digest, protocol=p, residual_qq=prepared['residual_qq'] if prepared is not None else None)
            write_data_json(root / 'diagnostics/summary.json', {'scope': 'presentation', 'figure_count': len(stems), 'table_count': len(table_paths), 'evaluation_bundle_manifest_sha256': bundle_hash})
    return target


### Complete artifact-only workflow


In [ ]:
def run_complete_evaluation_report(project_root, presentation_id, *, anchor_q_output, dimsim_output, random_control_output, bundle_path, bundle_manifest_sha256, derived_run_ids, derived_mode, source_hashes, source_digest, protocol=None, resolved_config=None):
    """Prepare or explicitly reuse derived data; never perform inference/scoring."""
    if derived_mode not in {'prepare', 'reuse'}:
        raise ValueError('derived_mode must be prepare or reuse')
    if not isinstance(derived_run_ids, dict) or set(derived_run_ids) != {'interpretation', 'presentation_data'}:
        raise ValueError('two explicit derived run IDs are required')
    base = Path(project_root).resolve() / 'outputs'
    interp = base / 'contextual-interpretation' / safe_run_id(derived_run_ids['interpretation'])
    prepared_path = base / 'contextual-interpretation-data' / safe_run_id(derived_run_ids['presentation_data'])
    report_path = base / 'contextual-evaluation-report' / safe_run_id(presentation_id)
    inputs = [Path(p).resolve(strict=True) for p in (anchor_q_output, dimsim_output, random_control_output, bundle_path)]
    for target in [report_path, interp, prepared_path] if derived_mode == 'prepare' else [report_path]:
        ensure_output_outside_inputs(target, inputs)
        if target.exists():
            raise FileExistsError(f'refusing to overwrite output: {target}')
        guard_report_output(target)
    p = protocol or load_protocol()
    apply_style(p)
    aq, ds = (load_evaluation_summary(inputs[0]), load_evaluation_summary(inputs[1]))
    check_paired_outputs(aq, ds)
    if file_sha256(inputs[3] / 'manifest.json') != bundle_manifest_sha256 or aq['manifest']['scientific_input_pins']['bundle_manifest_sha256'] != bundle_manifest_sha256:
        raise ValueError('explicit bundle identity differs from evaluation outputs')
    random = load_random_control_summary(inputs[2])
    if random['manifest']['scientific_input_pins']['bundle_manifest_sha256'] != bundle_manifest_sha256:
        raise ValueError('random control bundle identity mismatch')
    if derived_mode == 'prepare':
        run_saved_interpretation(dimsim_root=inputs[1], dimsim_manifest_sha256=file_sha256(inputs[1] / 'manifest.json'), bundle_path=inputs[3], bundle_manifest_sha256=bundle_manifest_sha256, project_root=project_root, run_id=interp.name, source_hashes=source_hashes)
        interpretation = read_finite_json(interp / 'data/interpretation.json')
        q_batch = load_saved_q_batch(inputs[0])
        prepared = build_interpretation_presentation_data(q_batch, interpretation)
        write_interpretation_presentation_data(project_root, prepared_path.name, prepared, input_roots=[interp, inputs[0]], source_hashes=source_hashes)
        del q_batch, interpretation, prepared
    else:
        im = read_finite_json(interp / 'manifest.json')
        verify_artifact_inventory(interp, 'contextual-interpretation', interp.name, set(im['required_artifacts']))
        interpretation = read_finite_json(interp / 'data/interpretation.json')
        saved = interpretation['config']['saved_inputs']
        if saved['bundle_manifest_sha256'] != bundle_manifest_sha256 or saved['dimsim_manifest_sha256'] != file_sha256(inputs[1] / 'manifest.json'):
            raise ValueError('reused interpretation has different source inputs')
        _, pm = load_interpretation_presentation_data(prepared_path)
        hashes = {r['sha256'] for r in pm['inputs']}
        if not {file_sha256(interp / 'manifest.json'), file_sha256(inputs[0] / 'manifest.json')} <= hashes:
            raise ValueError('reused prepared data has different source inputs')
        del interpretation
    return write_evaluation_report(project_root, presentation_id, anchor_q_output=inputs[0], dimsim_output=inputs[1], random_control_output=inputs[2], interpretation_data_output=prepared_path, source_hashes=source_hashes, source_digest=source_digest, protocol=p, resolved_config={**(resolved_config or {}), 'derived_mode': derived_mode, 'derived_run_ids': derived_run_ids, 'bundle_path': str(inputs[3]), 'bundle_manifest_sha256': bundle_manifest_sha256})


## 4. Select and verify artifacts

Each input manifest and artifact checksum is checked. Q and DIMSIM must describe the same bundle, models, seeds and checkpoints; the random control must use the same evaluation bundle. No source training directories are opened.


In [ ]:
def resolve_evaluation_runs(output_root, run_ids=None):
    """Resolve explicit runs or require exactly one completed candidate per analysis."""
    output_root = Path(output_root).resolve(strict=True)
    analyses = ("contextual-anchor-q", "contextual-dimsim", "contextual-random-control")
    if run_ids is not None and (not isinstance(run_ids, dict) or set(run_ids) != set(analyses)):
        raise ValueError("EVALUATION_RUN_IDS must specify all three analysis IDs")
    paths = {}
    for analysis in analyses:
        parent = output_root / analysis
        if run_ids is not None:
            paths[analysis] = parent / safe_run_id(run_ids[analysis])
            continue
        candidates = []
        for candidate in sorted(parent.iterdir()):
            manifest_path = candidate / "manifest.json"
            if not candidate.is_dir() or not manifest_path.is_file():
                continue
            manifest = read_finite_json(manifest_path)
            if (manifest.get("status") == "complete" and manifest.get("analysis") == analysis
                    and manifest.get("scope") == "numerical_outputs" and manifest.get("run_id") == candidate.name):
                candidates.append(candidate)
        if len(candidates) != 1:
            raise ValueError(
                f"{analysis}: found {len(candidates)} completed runs; set EVALUATION_RUN_IDS explicitly. "
                f"Candidates: {[path.name for path in candidates]}"
            )
        paths[analysis] = candidates[0]
    return paths


In [ ]:
INPUT_PATHS = resolve_evaluation_runs(EVALUATION_OUTPUT_ROOT, EVALUATION_RUN_IDS)
PROTOCOL = load_protocol()
apply_style(PROTOCOL)
SOURCE_HASHES = {
    **BUILD_METADATA["source_sha256"],
    **BUILD_METADATA["resource_sha256"],
    "visualization_notebook_builder": BUILD_METADATA["generator_sha256"],
}
SOURCE_DIGEST = hashlib.sha256(json.dumps(BUILD_METADATA, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
FONT_PROVENANCE = {}
for weight in ("normal", "bold"):
    filename = Path(font_manager.findfont(font_manager.FontProperties(family="Arial", weight=weight), fallback_to_default=False))
    FONT_PROVENANCE[weight] = {"path": str(filename), "sha256": file_sha256(filename)}
for analysis, path in INPUT_PATHS.items():
    print(analysis, path)


## 5. Prepare data and save six figures

In prepare mode, compute categorical fits, residual comparisons, top pairs, Q12/9/3 summaries and residual quantiles from saved scores. In reuse mode, read the exact completed derived runs. Save Anchor-Q transitions, Transition shares, Component shares, DIMSIM geometry, DIMSIM correlation and Residual Q–Q with tables and metadata. Existing outputs are never overwritten. For another rendering, select reuse and a new presentation ID.


In [ ]:
REPORT_PATH = run_complete_evaluation_report(
    OUTPUT_PROJECT_ROOT,
    PRESENTATION_ID,
    anchor_q_output=INPUT_PATHS["contextual-anchor-q"],
    dimsim_output=INPUT_PATHS["contextual-dimsim"],
    random_control_output=INPUT_PATHS["contextual-random-control"],
    bundle_path=BUNDLE_PATH,
    bundle_manifest_sha256=BUNDLE_MANIFEST_SHA256,
    derived_run_ids=DERIVED_RUN_IDS,
    derived_mode=DERIVED_MODE,
    source_hashes=SOURCE_HASHES,
    source_digest=SOURCE_DIGEST,
    resolved_config={
        "build_metadata": BUILD_METADATA,
        "selected_run_ids": {name: path.name for name, path in INPUT_PATHS.items()},
        "fonts": FONT_PROVENANCE,
    },
    protocol=PROTOCOL,
)
print("Saved report:", REPORT_PATH)
print("Figures:", REPORT_PATH / "figures")
print("Tables:", REPORT_PATH / "tables")
